# Menezes-Almeida et al. (2027) — analysis code

Python code used to process the apatite fission-track (AFT) data and to draw the graphs of:

> Menezes-Almeida, G., et al. (2027). *[title]*. Gondwana Research. https://doi.org/[DOI]

The sections follow the order in which the graphs and tables are cited in the article. Each section writes its script to a `.py` file (`%%writefile`); the last section runs all of them in the order required by their dependencies. Figures that are drawings by the authors (maps, Figs. 2 and 3 and the conceptual figures) are not reproduced here.

**Requirements:** Python ≥ 3.9 with `numpy`, `scipy`, `pandas`, `matplotlib`, `openpyxl`, `pyshp`, `shapely`, `pyproj` and (optional) `adjustText`:

```
pip install numpy scipy pandas matplotlib openpyxl pyshp shapely pyproj adjustText
```

**Data:** see the `data` folder of this Zenodo record and the paths in Section 0.

## 0. Setup

Paths used by all scripts. The input data (grain counts, track lengths, QTQt runs, expected t–T paths and the literature compilation) are in the `data` folder of this Zenodo record.

Script: `config.py`

In [ ]:
%%writefile config.py
"""Paths and data readers used by all scripts (relative to the folder of this notebook).
Download the record and keep its folder structure, or edit the paths below.

All AFT data of this study and the literature compilation are in ONE workbook,
data/AFT_data_Menezes-Almeida_2027.xlsx (sheets 'Samples', 'Grain counts',
'Track lengths', 'Dpar', 'Literature'). The functions below read it."""
import csv
import os
from functools import lru_cache

import numpy as np
import openpyxl

ROOT = os.path.abspath(os.getcwd())
DATA = os.path.join(ROOT, "data")
DATA_XLSX = os.path.join(DATA, "AFT_data_Menezes-Almeida_2027.xlsx")
SAMPLES_XLSX, SAMPLES_SHEET = DATA_XLSX, "Samples"     # 'Sample', 'Lon', 'Lat', 'Elevation (m)', 'Side' (W/E of the rift), ...
QTQT_DIRS = [os.path.join(DATA, "QTQt_runs")]          # one sub-folder per sample: QTQt .run and *_pred.txt files (chain files on request)
TT_DIR = os.path.join(DATA, "QTQt_expected_paths")     # 'TFS-xxx expected Tt file.txt' (expected path exported from QTQt)
RTJ_SHP = os.path.join(DATA, "shapes", "RTJ_rift")       # outline of the Reconcavo-Tucano-Jatoba rift
AREA_SHP = os.path.join(DATA, "shapes", "study_area")    # frame of the geological map (Fig. 1C)
COAST_SHP = os.path.join(DATA, "shapes", "geomorphology_IBGE_5M")  # 1:5,000,000 geomorphological map of Brazil (IBGE)
OUT = os.path.join(ROOT, "output")
os.makedirs(OUT, exist_ok=True)
LIT_TSV = os.path.join(OUT, "literature_AFT_data.tsv")  # tab-separated copy of sheet 'Literature' (written below)


@lru_cache(maxsize=None)
def _sheet(name):
    """Rows of one sheet as a list of dicts (header in row 1)."""
    wb = openpyxl.load_workbook(DATA_XLSX, read_only=True, data_only=True)
    rows = list(wb[name].iter_rows(values_only=True))
    hdr = [str(h).strip() if h is not None else "" for h in rows[0]]
    return hdr, [dict(zip(hdr, r)) for r in rows[1:] if any(v is not None for v in r)]


def sample_names():
    """Samples of this study, in ascending order."""
    return sorted(str(r["Sample"]) for r in _sheet("Samples")[1])


def read_sample(sample):
    """Ns, Ni per grain, rho_d (tracks/cm2), Nd and confined track lengths (um) of one sample."""
    s = next(r for r in _sheet("Samples")[1] if r["Sample"] == sample)
    g = [r for r in _sheet("Grain counts")[1] if r["Sample"] == sample]
    Ns = np.array([r["Ns"] for r in g], float)
    Ni = np.array([r["Ni"] for r in g], float)
    return Ns, Ni, float(s["rho_d (tracks/cm2)"]), float(s["Nd"]), read_lengths(sample)


def read_lengths(sample):
    return np.array([r["Length (µm)"] for r in _sheet("Track lengths")[1]
                     if r["Sample"] == sample and isinstance(r["Length (µm)"], (int, float))], float)


def _fmt(v):
    if v is None:
        return ""
    if isinstance(v, float):
        return str(int(v)) if v.is_integer() else repr(v)
    return str(v)


def export_literature():
    """Write sheet 'Literature' to LIT_TSV (read by the scripts as a tab-separated table)."""
    hdr, rows = _sheet("Literature")
    with open(LIT_TSV, "w", encoding="utf-8", newline="") as fh:
        w = csv.writer(fh, delimiter="\t", lineterminator="\n")
        w.writerow(hdr)
        for r in rows:
            w.writerow([_fmt(r[h]) for h in hdr])


export_literature()


## 1. Literature compilation: position relative to the RTJ rift (Supplementary Table S1)

Adds the column `RTJ_position` (W, E or inside) to the literature compilation by testing each sample against the rift outline.

Script: `prep_classify_RTJ_position.py`

In [ ]:
%%writefile prep_classify_RTJ_position.py
"""
classify_RTJ_position.py -- adds the column 'RTJ_position' to the literature table

Each literature sample is classified against the polygon of the
Reconcavo-Tucano-Jatoba rift (shapefile drawn in ArcGIS, SIRGAS 2000 geographic):

  inside                  sample inside the rift polygon (rift-fill / sedimentary rocks)
  W / E                   outside the rift, west or east of it along the same latitude
  between branches        outside the polygon but between two rift branches
  outside latitude range  north or south of the rift (W/E has no meaning there)

The test for W/E: a horizontal line through the sample is intersected with the
polygon; the sample is W if it lies west of the westernmost intersection, E if
east of the easternmost one. Checked on this study's samples: TFS-001..074 = E,
TFS-082 and TFS-101..120 = W, as on the geological map.

Needs:  pip install pyshp shapely
Run:    python classify_RTJ_position.py   (rewrites literature_AFT_data.tsv in place)
"""
import csv
import os

import shapefile                      # pyshp
from shapely.geometry import LineString, Point, shape

from config import RTJ_SHP as SHAPE

POLY = shape(shapefile.Reader(SHAPE).shape(0).__geo_interface__)


def rtj_position(lon, lat):
    if POLY.contains(Point(lon, lat)):
        return "inside"
    cut = POLY.intersection(LineString([(-60, lat), (-20, lat)]))
    if cut.is_empty:
        return "outside latitude range"
    xs = [c[0] for g in getattr(cut, "geoms", [cut]) for c in g.coords]
    if lon < min(xs):
        return "W"
    if lon > max(xs):
        return "E"
    return "between branches"


def main():
    """Classify each published sample and write the result to column 'RTJ_position'
    of sheet 'Literature' in the data workbook."""
    import openpyxl
    import config
    wb = openpyxl.load_workbook(config.DATA_XLSX)
    ws = wb["Literature"]
    hdr = [c.value for c in ws[1]]
    if "RTJ_position" in hdr:
        col = hdr.index("RTJ_position") + 1
    else:
        col = len(hdr) + 1
        ws.cell(1, col, "RTJ_position")
    ix, iy = hdr.index("coord_X") + 1, hdr.index("coord_Y") + 1
    fl = lambda v: float(str(v).replace(",", "."))
    counts = {}
    for i in range(2, ws.max_row + 1):
        x, y = ws.cell(i, ix).value, ws.cell(i, iy).value
        if x is None or y is None:
            continue
        pos = rtj_position(fl(x), fl(y))
        ws.cell(i, col, pos)
        counts[pos] = counts.get(pos, 0) + 1
    wb.save(config.DATA_XLSX)
    config._sheet.cache_clear()
    config.export_literature()
    print(counts)


if __name__ == "__main__":
    main()


## 2. Radial plots (Supplementary Fig. S1)

Central ages, P(χ²) and dispersion recalculated from the single-grain counts (ζ = 266.85 ± 6.42), and one radial plot per sample (Galbraith, 1990).

Script: `figS1_radial_plots.py`

In [ ]:
%%writefile figS1_radial_plots.py
"""
3_radial_plots.py  --  Galbraith radial plots for the AFT samples (this study)
==============================================================================

Reads the single-grain counts (Ns, Ni) straight from the counting spreadsheets
(sheets 'Grain counts' and 'Samples' of the data workbook) and draws one radial plot per
sample (Galbraith 1988, 1990), plus a multi-panel figure with every sample
grouped by crustal domain. The script does its own statistics, so it does not
depend on IsoplotR.

What is plotted (Galbraith 1990, Galbraith & Laslett 1993)
---------------------------------------------------------
  z_j   = ln[(Ns_j + 0.5) / (Ni_j + 0.5)]             log track-density ratio
  s_j   = sqrt[1/(Ns_j + 0.5) + 1/(Ni_j + 0.5)]       its standard error
  x_j   = 1 / s_j                                      precision
  y_j   = (z_j - z0) / s_j                             standardised estimate
  z0    = ln of the central ratio (random-effects estimate)

  The single-grain age of each point is read on the arc by projecting a line
  from the origin through the point. The grey band is y = +/-2: if the grains
  share one age (within Poisson error), about 95% of them should fall inside it.
  Ages:  t = (1/lambda) ln(1 + lambda * zeta * rho_d * (Ns/Ni) / 2)

  Central age, its 1 sigma error (with the Nd term), P(chi2) and dispersion
  follow Galbraith & Laslett (1993). The values match the spreadsheet / QTQt
  input ages (the zeta error is left out, the same convention as the
  spreadsheets; set ZETA_SE to include it).

Outputs (in OUT_DIR)
--------------------
  radial_TFS-XXX.pdf / .png          one plot per sample
  ALL_SAMPLES_radial_plots.pdf/.png  multi-panel figure, grouped by domain
  AFT_single_grain_ages.csv          Ns, Ni, single-grain age +/-1 sigma, z, x, y
  AFT_central_age_summary.csv        n, central age, P(chi2), dispersion, and a
                                     check against the age written in the sheet

How to use
----------
  1. Check the paths in config.py.
  2. python 3_radial_plots.py
"""

import glob
import os
import re

import numpy as np
import openpyxl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Polygon
from matplotlib.ticker import MaxNLocator
from scipy.stats import chi2

plt.rcParams["pdf.fonttype"] = 42          # editable text in CorelDRAW
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.formatter.use_locale"] = False   # decimal point, never comma

# ---------------------------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------------------------
from config import sample_names, read_sample, OUT as OUT_DIR

ZETA = 266.85          # zeta calibration factor (Zeta-GLsSDsM40.xlsx)
ZETA_SE = 0.0          # 1 sigma of zeta; 0 = same convention as the spreadsheets
LAMBDA = 1.55125e-10   # 238U total decay constant (1/yr)
EXCLUDE = []           # samples to skip, e.g. ["TFS-018"]
AGE_CHECK_TOL_MA = 0.5
MAX_ARC_DEG = 65       # arc ends no steeper than this (keeps plots wide, not tall) # warn if recalculated central age differs from the sheet

# Tectonic province of each sample (position relative to the SFC / Borborema
# Province boundary on the geological map; same list in every script)
SFC_SAMPLES = {1, 5, 9, 10, 16, 18, 21, 101, 104, 106, 108, 112, 114, 116, 120}   # all others: Borborema
DOMAIN_COLORS = {                       # SFC red / Borborema blue (as in the kriging map)
    "Borborema Province": "#3f6f8f",
    "São Francisco Craton": "#a6423a",
}
DOMAIN_ORDER = ["São Francisco Craton", "Borborema Province"]

# Layout of the multi-panel figure
N_COLS = 5


def classify_domain(num):
    return "São Francisco Craton" if num in SFC_SAMPLES else "Borborema Province"


# ---------------------------------------------------------------------------
# READING THE COUNTING SPREADSHEETS
# ---------------------------------------------------------------------------
def read_counts(sample):
    """Ns, Ni per grain, rho_d and Nd of one sample (sheets 'Grain counts' and 'Samples')."""
    Ns, Ni, rhod, Nd, _ = read_sample(sample)
    return Ns, Ni, rhod, Nd, None


# ---------------------------------------------------------------------------
# STATISTICS
# ---------------------------------------------------------------------------
def ratio_to_age(ratio, rhod):
    return np.log1p(LAMBDA * ZETA * rhod * 0.5 * ratio) / LAMBDA / 1e6


def central_age(Ns, Ni, rhod, Nd):
    """Galbraith & Laslett (1993) random-effects central age."""
    m = Ns + Ni
    y = Ns / m
    sig = 0.6
    w = np.ones_like(m)
    for _ in range(500):
        eta = np.sum(w * y) / np.sum(w)
        w = m / (eta * (1 - eta) + (m - 1) * eta ** 2 * (1 - eta) ** 2 * sig ** 2)
        sig_new = sig * np.sqrt(np.sum(w ** 2 * (y - eta) ** 2) / np.sum(w))
        if abs(sig_new - sig) < 1e-9:
            sig = sig_new
            break
        sig = sig_new
    ratio = eta / (1 - eta)
    se_eta = 1 / np.sqrt(np.sum(w))
    t = ratio_to_age(ratio, rhod)
    rel = np.sqrt((se_eta / (eta * (1 - eta))) ** 2 + 1 / Nd
                  + (ZETA_SE / ZETA) ** 2)
    Ns_t, Ni_t = Ns.sum(), Ni.sum()
    X2 = np.sum((Ns * Ni_t - Ni * Ns_t) ** 2 / (Ns + Ni)) / (Ns_t * Ni_t)
    P = 1 - chi2.cdf(X2, len(Ns) - 1)
    return dict(t=t, se=t * rel, z0=np.log(ratio), disp=sig if sig > 1e-4 else 0.0,
                X2=X2, P=P, pooled=ratio_to_age(Ns_t / Ni_t, rhod))


def grain_stats(Ns, Ni, rhod, z0):
    z = np.log((Ns + 0.5) / (Ni + 0.5))
    s = np.sqrt(1 / (Ns + 0.5) + 1 / (Ni + 0.5))
    age = ratio_to_age(np.exp(z), rhod)
    # 1 sigma single-grain age error (Poisson on Ns and Ni)
    age_se = age * np.sqrt(1 / np.maximum(Ns, 0.5) + 1 / np.maximum(Ni, 0.5))
    return dict(z=z, s=s, x=1 / s, y=(z - z0) / s, age=age, age_se=age_se)


# ---------------------------------------------------------------------------
# DRAWING ONE RADIAL PLOT
# ---------------------------------------------------------------------------
def nice_age_ticks(tmin, tmax, n=6):
    loc = MaxNLocator(nbins=n, steps=[1, 2, 2.5, 5, 10])
    ticks = [v for v in loc.tick_values(tmin, tmax) if tmin <= v <= tmax and v > 0]
    return ticks


def radial_plot(ax, sample, g, c, rhod, color, compact=False, head_fig=None):
    """Draw a radial plot on `ax`. Plot units: u = k*x (precision), v = y."""
    x, y, z0 = g["x"], g["y"], c["z0"]
    ylim_data = max(np.abs(y).max(), 2.5)

    # age range on the arc: covers every grain
    t_g = g["age"]
    tmin, tmax = max(t_g.min(), 1.0), t_g.max()
    tmin, tmax = tmin / 1.06, tmax * 1.06          # small margin, in log space
    z_of = lambda t: np.log(np.expm1(LAMBDA * t * 1e6) / (LAMBDA * ZETA * rhod * 0.5))
    dz_max = max(abs(z_of(tmin) - z0), abs(z_of(tmax) - z0))

    # k stretches precision against the standardised axis: wide enough to separate
    # the grains, and to keep the arc ends within +/-MAX_ARC_DEG of the horizontal
    k = max(1.9 * ylim_data / x.max(), dz_max / np.tan(np.radians(MAX_ARC_DEG)))
    u = k * x
    R = 1.08 * np.sqrt(u ** 2 + y ** 2).max()   # arc radius, just outside the data

    def theta(t):  # angle of the ray that carries age t
        return np.arctan((z_of(t) - z0) / k)
    th_lo, th_hi = theta(tmin), theta(tmax)

    # +/-2 band, clipped by the arc
    u2 = np.sqrt(R ** 2 - 4.0)
    arc_band = np.linspace(np.arcsin(-2 / R), np.arcsin(2 / R), 40)
    band = [(0, -2), (u2, -2)] + [(R * np.cos(a), R * np.sin(a)) for a in arc_band] + [(0, 2)]
    ax.add_patch(Polygon(band, closed=True, fc="#e8e4da", ec="none", zorder=0))

    # arc and age ticks
    a = np.linspace(th_lo, th_hi, 200)
    ax.plot(R * np.cos(a), R * np.sin(a), color="#1b1b1b", lw=0.9, zorder=2)
    tl = 0.025 * R
    fs_tick = 6.5 if compact else 8
    for t in nice_age_ticks(tmin, tmax, 5 if compact else 7):
        th = theta(t)
        ax.plot([R * np.cos(th), (R + tl) * np.cos(th)],
                [R * np.sin(th), (R + tl) * np.sin(th)], color="#1b1b1b", lw=0.8)
        ax.text((R + 2.6 * tl) * np.cos(th), (R + 2.6 * tl) * np.sin(th), f"{t:g}",
                fontsize=fs_tick, ha="left", va="center", rotation=np.degrees(th),
                rotation_mode="anchor")
    if not compact:
        ax.text((R + 3 * tl) * np.cos(th_hi), (R + 3 * tl) * np.sin(th_hi) + 0.09 * R,
                "Age (Ma)", fontsize=8.5, ha="left", va="bottom", style="italic")

    # central-age line
    ax.plot([0, R], [0, 0], color=color, lw=1.0, ls=(0, (4, 2)), zorder=1)

    # grains (an open marker = grain with Ns = 0)
    zero = g["Ns"] == 0
    ms = 22 if compact else 38
    ax.scatter(u[~zero], y[~zero], s=ms, c=color, edgecolors="#1b1b1b", lw=0.5, zorder=4)
    if zero.any():
        ax.scatter(u[zero], y[zero], s=ms, facecolors="white", edgecolors=color, lw=1.0, zorder=4)

    # left axis: standardised estimate, only -2..+2
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines["left"].set_bounds(-2, 2)
    ax.set_yticks([-2, 0, 2])
    ax.set_yticklabels(["−2", "0", "+2"], fontsize=fs_tick)
    if not compact:
        ax.set_ylabel("Standardised estimate", fontsize=9)

    # bottom axis: precision (1/sigma) and, below it, relative error (%)
    xt = [v for v in MaxNLocator(nbins=4, integer=True).tick_values(0, x.max()) if 0 <= v <= x.max() * 1.05]
    ax.spines["bottom"].set_bounds(0, k * xt[-1])
    ax.set_xticks([k * v for v in xt])
    ax.set_xticklabels([f"{v:g}" for v in xt], fontsize=fs_tick)
    if not compact:
        ax.set_xlabel("Precision (t/σ)", fontsize=9, labelpad=1)
        ax.xaxis.set_label_coords(0.0, -0.075, transform=ax.transAxes)
        ax.xaxis.label.set_ha("left")
        # second scale under the precision axis: relative error (%) = 100 / precision
        tr = ax.get_xaxis_transform()   # x in data units, y in axes fraction
        y0 = -0.14
        pct = [p for p in [100, 50, 25, 15, 10, 7.5, 5] if 100 / p <= xt[-1] * 1.001]
        ax.plot([0, k * xt[-1]], [y0, y0], transform=tr, color="#1b1b1b", lw=0.8, clip_on=False)
        last = -1e9
        for p_ in pct:
            uu = k * 100 / p_
            ax.plot([uu, uu], [y0, y0 - 0.015], transform=tr, color="#1b1b1b", lw=0.8, clip_on=False)
            if (uu - last) / (R * 1.22) < 0.11:
                continue
            last = uu
            ax.text(uu, y0 - 0.025, f"{p_:g}", transform=tr, ha="center", va="top",
                    fontsize=fs_tick, clip_on=False)
        ax.text(0, y0 - 0.075, "Relative error (%)", transform=tr, ha="left", va="top",
                fontsize=9, clip_on=False)

    # limits, equal aspect so the arc is a circle
    ax.set_xlim(0, R * 1.22)
    ax.set_ylim(min(R * np.sin(th_lo), -2.5) - 0.08 * R, max(R * np.sin(th_hi), 2.5) + (0.32 if compact else 0.14) * R)
    ax.set_aspect("equal")

    # summary text
    disp = 100 * c["disp"]
    pflag = " *" if c["P"] < 0.05 else ""
    fs_head = 8 if compact else 10.5
    if head_fig is not None:
        head_fig.text(0.04, 0.965, sample, ha="left", va="top", fontsize=13, fontweight="bold")
        head_fig.text(0.20, 0.965, f"{c['t']:.1f} ± {c['se']:.1f} Ma (±1σ)", ha="left", va="top", fontsize=12)
        head_fig.text(0.04, 0.915, f"n = {len(x)}   P(χ²) = {c['P']:.2f}{pflag}   Disp. = {disp:.0f}%",
                      ha="left", va="top", fontsize=10, color="#444444")
    else:
        head = ax.text(0, 1.0, sample, transform=ax.transAxes, ha="left", va="bottom",
                       fontsize=fs_head, fontweight="bold")
        ax.annotate(f"{c['t']:.1f} ± {c['se']:.1f} Ma", xy=(1, 0), xycoords=head,
                    xytext=(5, 0), textcoords="offset points", ha="left", va="bottom",
                    fontsize=fs_head)
        ax.text(0, 0.995, f"n = {len(x)}   P(χ²) = {c['P']:.2f}{pflag}   Disp. = {disp:.0f}%",
                transform=ax.transAxes, ha="left", va="top", fontsize=fs_tick, color="#444444")
    return ax


# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------
def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    files = sample_names()

    samples = []
    for f in files:
        sample = os.path.splitext(os.path.basename(f))[0]
        if sample in EXCLUDE:
            continue
        num = int(re.search(r"(\d+)", sample).group(1))
        Ns, Ni, rhod, Nd, sheet_age = read_counts(f)
        c = central_age(Ns, Ni, rhod, Nd)
        g = grain_stats(Ns, Ni, rhod, c["z0"])
        g["Ns"], g["Ni"] = Ns, Ni
        sheet_t = float(re.match(r"\s*([\d.]+)", sheet_age).group(1)) if sheet_age else np.nan
        note = ""
        if np.isfinite(sheet_t) and abs(sheet_t - c["t"]) > AGE_CHECK_TOL_MA:
            note = f"sheet age {sheet_age} differs from recalculated {c['t']:.2f}"
            print(f"[warning] {sample}: {note}")
        samples.append(dict(sample=sample, num=num, domain=classify_domain(num), rhod=rhod,
                            Nd=Nd, c=c, g=g, sheet_age=sheet_age, note=note))

    # --- tables ---
    with open(os.path.join(OUT_DIR, "AFT_single_grain_ages.csv"), "w", encoding="utf-8") as fh:
        fh.write("sample,domain,grain,Ns,Ni,age_Ma,age_1sigma_Ma,z,precision_x,standardised_y\n")
        for s in samples:
            g = s["g"]
            for j in range(len(g["x"])):
                fh.write(f"{s['sample']},{s['domain']},{j + 1},{g['Ns'][j]:.0f},{g['Ni'][j]:.0f},"
                         f"{g['age'][j]:.2f},{g['age_se'][j]:.2f},{g['z'][j]:.4f},"
                         f"{g['x'][j]:.3f},{g['y'][j]:.3f}\n")
    with open(os.path.join(OUT_DIR, "AFT_central_age_summary.csv"), "w", encoding="utf-8") as fh:
        fh.write("sample,domain,n_grains,Ns_total,Ni_total,rho_d,Nd,central_age_Ma,1sigma_Ma,"
                 "pooled_age_Ma,chi2,P_chi2,dispersion_pct,min_grain_age_Ma,max_grain_age_Ma,"
                 "sheet_age,note\n")
        for s in samples:
            c, g = s["c"], s["g"]
            fh.write(f"{s['sample']},{s['domain']},{len(g['x'])},{g['Ns'].sum():.0f},"
                     f"{g['Ni'].sum():.0f},{s['rhod']:.0f},{s['Nd']:.1f},{c['t']:.1f},{c['se']:.1f},"
                     f"{c['pooled']:.1f},{c['X2']:.2f},{c['P']:.3f},{100 * c['disp']:.0f},"
                     f"{g['age'].min():.1f},{g['age'].max():.1f},{s['sheet_age'] or ''},{s['note']}\n")

    # --- one figure per sample ---
    for s in samples:
        fig = plt.figure(figsize=(6.4, 5.4))
        ax = fig.add_axes([0.10, 0.20, 0.72, 0.62])
        radial_plot(ax, s["sample"], s["g"], s["c"], s["rhod"], DOMAIN_COLORS[s["domain"]], head_fig=fig)
        for ext in ("pdf", "png"):
            fig.savefig(os.path.join(OUT_DIR, f"radial_{s['sample']}.{ext}"), dpi=300)
        plt.close(fig)

    # --- multi-panel figure, grouped by domain ---
    order = sorted(samples, key=lambda s: (DOMAIN_ORDER.index(s["domain"]), s["num"]))
    n = len(order)
    n_rows = int(np.ceil(n / N_COLS))
    fig, axes = plt.subplots(n_rows, N_COLS, figsize=(3.1 * N_COLS, 2.35 * n_rows))
    axes = np.atleast_2d(axes)
    for i, ax in enumerate(axes.flat):
        if i >= n:
            ax.axis("off")
            continue
        s = order[i]
        radial_plot(ax, s["sample"], s["g"], s["c"], s["rhod"], DOMAIN_COLORS[s["domain"]],
                    compact=True)
    handles = [Line2D([0], [0], marker="o", ls="", mfc=DOMAIN_COLORS[d], mec="#1b1b1b", ms=7, label=d)
               for d in DOMAIN_ORDER if any(s["domain"] == d for s in samples)]
    handles += [Polygon([(0, 0)], fc="#e8e4da", label="±2 band around the central age"),
                Line2D([0], [0], color="#555555", lw=1, ls=(0, (4, 2)), label="central age")]
    fig.legend(handles=handles, loc="lower center", ncol=len(handles), frameon=False, fontsize=9,
               bbox_to_anchor=(0.5, 0.0))
    fig.text(0.01, 0.003, "Arc: single-grain age (Ma); x axis: precision (t/σ); y axis: standardised estimate. Central age ±1σ. * P(χ²) < 0.05 (over-dispersed). Central ages after Galbraith & Laslett (1993); "
             f"ζ = {ZETA} (1σ error without ζ uncertainty).", fontsize=7.5, color="#555555")
    fig.subplots_adjust(left=0.03, right=0.99, top=0.99, bottom=0.05, wspace=0.18, hspace=0.42)
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(OUT_DIR, f"ALL_SAMPLES_radial_plots.{ext}"), dpi=300)
    plt.close(fig)

    print(f"{len(samples)} samples -> {OUT_DIR}")
    for s in order:
        c = s["c"]
        print(f"  {s['sample']:8s} {c['t']:7.1f} ± {c['se']:5.1f} Ma   P = {c['P']:.3f}   "
              f"disp = {100 * c['disp']:3.0f}%   {s['domain']}")


if __name__ == "__main__":
    main()


## 3. Age–elevation plot (Supplementary Fig. S2)

AFT central age versus elevation for this study and for the published samples inside the study area. Uses the readers of `fig06_half_boomerang.py`.

Script: `figS2_age_elevation.py`

In [ ]:
%%writefile figS2_age_elevation.py
"""
5_age_elevation.py  --  AFT central age vs sample elevation
============================================================

Same symbology as the boomerang plots (4_boomerang_plots.py):
  colour = tectonic province (São Francisco Craton red / Borborema Province blue)
  shape  = side of the Recôncavo-Tucano-Jatobá rift (square = W, circle = E)
  dashed border = sample without enough confined tracks (< 50) for thermal
                  modelling (TFS-018, 029, 046) or without valid lengths (TFS-021)

  (a) this study (all 27 samples)
  (b) this study + literature inside the study area (open symbols; square = W,
      circle = E of the rift, x = inside the rift)

Inputs
------
  Central ages      recalculated from the counting spreadsheets (same as scripts 3 and 4)
  Elevations        data workbook, sheet 'Samples', column 'Elevation (m)'
  Literature        literature_AFT_data.tsv (column 'Elevation', 'RTJ_position')
  Study-area shape  'nova_area'

Turner et al. (2008) samples all carry 200 m in the compilation (a placeholder,
not a measured elevation), so they are left out of panel (b); see EXCLUDE_REFS.
"""

import glob
import importlib.util
import os
import re

import numpy as np
import openpyxl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import shapefile
from shapely.geometry import Point, shape

HERE = os.path.dirname(os.path.abspath(__file__))
spec = importlib.util.spec_from_file_location("bp", os.path.join(HERE, "fig06_half_boomerang.py"))
bp = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bp)

from config import SAMPLES_XLSX, SAMPLES_SHEET, AREA_SHP as STUDY_AREA_SHP
OUT_DIR = bp.OUT_DIR
NOT_MODELLED = {"TFS-018", "TFS-021", "TFS-029", "TFS-046"}
EXCLUDE_REFS = {"Turner et al. (2008)"}     # placeholder elevations (all 200 m)
LABEL_SAMPLES = True


def read_elevations(path, sheet):
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    rows = wb[sheet].iter_rows(values_only=True)
    hdr = [str(h).strip() if h is not None else "" for h in next(rows)]
    i_s, i_e = hdr.index("Sample"), hdr.index("Elevation (m)")
    out = {}
    for r in rows:
        if r[i_s] and isinstance(r[i_e], (int, float)):
            out[str(r[i_s]).strip()] = float(r[i_e])
    return out


def read_literature(path, area):
    fl = lambda s: float(str(s).replace(",", ".")) if str(s).strip() not in ("", "-") else np.nan
    out = []
    with open(path, encoding="utf-8", errors="ignore") as fh:
        hdr = fh.readline().rstrip("\n").split("\t")
        ix = {h: i for i, h in enumerate(hdr)}
        for line in fh:
            c = line.rstrip("\n").split("\t")
            if len(c) < len(hdr) - 1:
                continue
            x, y = fl(c[ix["coord_X"]]), fl(c[ix["coord_Y"]])
            age, elev = fl(c[ix["AFT_age"]]), fl(c[ix["Elevation"]])
            ref = c[ix["reference"]].strip()
            if np.isnan(age) or np.isnan(elev) or ref in EXCLUDE_REFS or not area.contains(Point(x, y)):
                continue
            out.append(dict(ref=ref, age=age, err=fl(c[ix["Age error"]]), elev=elev,
                            pos=c[ix["RTJ_position"]].strip() if "RTJ_position" in ix else "?"))
    return out


def draw_own(ax, s, label=True):
    c = bp.DOMAIN_COLORS[s["domain"]]
    ax.errorbar(s["age"], s["elev"], xerr=s["age_se"], fmt="none", ecolor=c, elinewidth=0.8,
                capsize=0, alpha=0.6, zorder=3)
    dashed = s["sample"] in NOT_MODELLED
    ax.scatter(s["age"], s["elev"], s=62, marker=bp.SIDE_MARKERS[s["side"]], c=c, edgecolors="#1b1b1b",
               linewidths=0.6 if dashed else 0.7, linestyles=(0, (1.5, 1.2)) if dashed else "solid", zorder=5)
    if label:
        return ax.text(s["age"] + 3, s["elev"] + 8, str(s["num"]).zfill(3), fontsize=6.5,
                       color="#444444", zorder=7)


def main():
    elev = read_elevations(SAMPLES_XLSX, SAMPLES_SHEET)
    area = shape(shapefile.Reader(STUDY_AREA_SHP).shape(0).__geo_interface__)
    lit = read_literature(bp.LIT_TSV, area)

    samples = []
    for f in bp.sample_names():
        name = os.path.splitext(os.path.basename(f))[0]
        if name not in elev:
            print(f"[warning] no elevation for {name}")
            continue
        num = int(re.search(r"(\d+)", name).group(1))
        Ns, Ni, rhod, Nd, _ = bp.read_sheet(f)
        age, se = bp.central_age(Ns, Ni, rhod, Nd)
        samples.append(dict(sample=name, num=num, domain=bp.classify_domain(num),
                            side="west" if num in bp.WEST_OF_TUCANO else "east",
                            age=age, age_se=se, elev=elev[name]))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6.2), sharey=True)
    labels = [draw_own(ax1, s, LABEL_SAMPLES) for s in samples]
    for pos, mk in (("W", "s"), ("E", "o")):
        pts = [l for l in lit if l["pos"] == pos]
        ax2.scatter([l["age"] for l in pts], [l["elev"] for l in pts], s=22, marker=mk,
                    facecolors="white", edgecolors="#777777", linewidths=0.8, zorder=2)
    inside = [l for l in lit if l["pos"] not in ("W", "E")]
    ax2.scatter([l["age"] for l in inside], [l["elev"] for l in inside], s=26, marker="x",
                c="#9a9a9a", linewidths=0.9, zorder=2)
    for s in samples:
        draw_own(ax2, s, label=False)

    xmax = max([s["age"] + s["age_se"] for s in samples] + [l["age"] for l in lit]) * 1.05
    ymax = max([s["elev"] for s in samples] + [l["elev"] for l in lit]) * 1.08
    for ax, title in ((ax1, "(a) This study"), (ax2, "(b) This study and literature inside the study area")):
        ax.set_xlim(0, xmax)
        ax.set_ylim(min(0, min([l["elev"] for l in lit] + [0])) - 20, ymax)
        ax.set_xlabel("AFT central age (Ma)", fontsize=10)
        ax.grid(True, color="#dddddd", lw=0.5, zorder=0)
        ax.spines[["top", "right"]].set_visible(False)
        ax.tick_params(labelsize=8.5)
        ax.set_title(title, loc="left", fontsize=10.5, fontweight="bold")
    ax1.set_ylabel("Elevation (m a.s.l.)", fontsize=10)

    labels = [t for t in labels if t is not None]
    if labels:
        try:
            from adjustText import adjust_text
            adjust_text(labels, ax=ax1, expand=(1.3, 1.6),
                        arrowprops=dict(arrowstyle="-", color="#999999", lw=0.5))
        except ImportError:
            pass

    h = []
    for d in bp.DOMAIN_ORDER:
        for side in ("west", "east"):
            n = sum(s["domain"] == d and s["side"] == side for s in samples)
            if n:
                h.append(Line2D([0], [0], marker=bp.SIDE_MARKERS[side], ls="", mfc=bp.DOMAIN_COLORS[d],
                                mec="#1b1b1b", ms=8, label=f"{d}, {bp.SIDE_LABELS[side]} (n = {n})"))
    h.append(plt.scatter([], [], s=62, marker="o", c="#bbbbbb", edgecolors="#1b1b1b", linewidths=0.6,
                         linestyles=(0, (1.5, 1.2)), label="Not modelled (< 50 confined tracks / no valid lengths)"))
    for pos, mk, lab in (("W", "s", "Literature W of RTJ"), ("E", "o", "Literature E of RTJ")):
        h.append(Line2D([0], [0], ls="", marker=mk, mfc="white", mec="#777777", ms=6,
                        label=f"{lab} (n = {sum(l['pos'] == pos for l in lit)})"))
    h.append(Line2D([0], [0], ls="", marker="x", color="#9a9a9a", ms=6,
                    label=f"Literature inside the RTJ rift (n = {len(inside)})"))
    fig.legend(handles=h, loc="lower center", ncol=3, frameon=False, fontsize=8.3, bbox_to_anchor=(0.5, 0.03))
    fig.text(0.5, 0.005, "Horizontal bars: central age ±1σ (this study). Literature: samples inside the study-area "
             "polygon with published elevation; Turner et al. (2008) excluded (placeholder elevations).",
             fontsize=7, color="#555555", ha="center")
    fig.subplots_adjust(left=0.06, right=0.99, top=0.93, bottom=0.25, wspace=0.08)
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(OUT_DIR, f"age_elevation.{ext}"), dpi=300)
    plt.close(fig)

    # quick numbers: age-elevation correlation per side (this study)
    for side in ("west", "east"):
        a = np.array([s["age"] for s in samples if s["side"] == side])
        e = np.array([s["elev"] for s in samples if s["side"] == side])
        if len(a) > 2:
            r = np.corrcoef(a, e)[0, 1]
            print(f"{side}: n = {len(a)}, elevation {e.min():.0f}-{e.max():.0f} m, Pearson r(age, elev) = {r:.2f}")
    a = np.array([s["age"] for s in samples]); e = np.array([s["elev"] for s in samples])
    print(f"all: n = {len(a)}, r = {np.corrcoef(a, e)[0, 1]:.2f}; literature in panel (b): {len(lit)}")


if __name__ == "__main__":
    main()


## 4. Confined track-length histograms (Supplementary Fig. S3)

Histograms and kernel density curves for the 23 samples with at least 50 confined tracks.

Script: `figS3_track_length_histograms.py`

In [ ]:
%%writefile figS3_track_length_histograms.py
"""Confined track-length histograms (Supplementary Fig. S3) for the samples with >= 50 confined tracks."""
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from config import read_lengths, OUT

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Montserrat", "DejaVu Sans", "Arial"]
plt.rcParams["pdf.fonttype"] = 42

SAMPLES = ["TFS-001", "TFS-005", "TFS-009", "TFS-010", "TFS-016", "TFS-035", "TFS-039", "TFS-049",
           "TFS-051", "TFS-053", "TFS-055", "TFS-061", "TFS-068", "TFS-074", "TFS-082", "TFS-101",
           "TFS-104", "TFS-106", "TFS-108", "TFS-112", "TFS-114", "TFS-116", "TFS-120"]
OUT_DIR = os.path.join(OUT, "track_length_histograms")
os.makedirs(OUT_DIR, exist_ok=True)

for sample_name in SAMPLES:
    data = pd.Series(read_lengths(sample_name))
    n = len(data)
    fig = plt.figure(figsize=(8, 6))
    bin_edges = list(range(0, 19, 1))
    plt.hist(data, bins=bin_edges, color="#F5F5DC", edgecolor="black")
    kde = gaussian_kde(data)
    x_grid = np.linspace(data.min(), data.max(), 1000)
    plt.plot(x_grid, kde(x_grid) * n * (bin_edges[1] - bin_edges[0]), color="#8B0000", linewidth=2.5)
    plt.xlabel("Confined track length (µm)", fontsize=12)
    plt.ylabel("Frequency", fontsize=12)
    plt.title(sample_name, fontsize=20, loc="left", fontweight="bold")
    plt.xlim(0, 18)
    ax = plt.gca()
    ax.set_xticks(range(0, 19, 2)); ax.set_xticks(range(1, 18, 2), minor=True)
    plt.text(0.02, 0.95, f"{sample_name}\nn = {n}\nMTL = {np.mean(data):.1f}\n$\\sigma$ = {np.std(data):.1f}",
             transform=ax.transAxes, fontsize=12, fontweight="bold", va="top", ha="left",
             bbox=dict(facecolor="white", edgecolor="white", alpha=0.8))
    plt.tight_layout()
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(OUT_DIR, f"{sample_name}_histogram.{ext}"), dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(sample_name, "n =", n)


## 5. Cooling histories, step 1: breakpoint segmentation (Table 3; Supplementary Fig. S5)

Piecewise-linear fit of the QTQt expected paths, with the number of segments chosen by the Bayesian Information Criterion, and definition of the main cooling episode.

Script: `figS5a_breakpoints.py`

In [ ]:
%%writefile figS5a_breakpoints.py
"""
QTQt Expected Model - data-driven segmentation of t-T paths (breakpoint detection), v3
=====================================================================================
Step 1 of 2 (run before 2_posterior_rates.py). Input: QTQt expected Tt files.

Idea (analogous to the matched-filter approach of Spector & Grant, 1970, where
straight-line segments fitted to the radially averaged power spectrum define
source ensembles): instead of cutting every t-T path at fixed ages, fit a
CONTINUOUS piecewise-linear model to the expected path and let the changes of
slope (inflection points) define the segments. Each segment is a thermal regime;
each breakpoint is the age of a change of regime.

Number of breakpoints (objective)
---------------------------------
For k = 0..MAX_BREAKS the breakpoint ages are optimised by least squares
(differential evolution, fixed seed -> reproducible) and scored with the BIC:
    BIC = n * ln(RSS_eff / n) + (2k + 2) * ln(n)
The expected path is a smooth posterior mean, not noisy data, so the residual is
floored at a temperature resolution SIGMA_FLOOR_C:
    RSS_eff = max(RSS, n * SIGMA_FLOOR_C^2)
i.e. misfits below ~SIGMA_FLOOR_C are not treated as signal. The default (3 degC)
is still far below the 95% credible-interval half-widths of the paths (10-50 degC).
Because the least-squares fits do not depend on SIGMA_FLOOR_C, a sensitivity
table (number of breakpoints and onset age for SENSITIVITY_SIGMAS) is produced
at no extra cost - report it in the Supplement.
Guards: minimum segment duration, minimum slope change between adjacent segments.

Main cooling episode (changed in v2)
------------------------------------
For each cooling segment, the temperature drop INSIDE the APAZ (60-120 degC) is
computed (overlap of [T_end, T_start] with the APAZ). The segment with the largest
in-APAZ drop is the main cooling episode. This replaces "largest total drop",
which favoured long segments running through and beyond the APAZ.

Onset of main cooling (changed in v3)
-------------------------------------
Two measures are reported, because they answer different questions:
- peak_T_age_Ma: age of the maximum temperature of the EXPECTED PATH older than the
  end of the main episode = start of net cooling. Read from the path itself, so it
  does not depend on the piecewise fit or on SIGMA_FLOOR_C. If this peak lies within
  MIN_SEGMENT_MYR of the oldest point of the path, the onset is CENSORED: the sample
  was already cooling when the modelled time range begins (onset_type =
  "censored (>= X Ma)", onset_min_Ma = X). In v2 censoring was decided from the fit,
  which flagged samples as censored whenever a coarse fit merged an early heating
  phase into the main episode (e.g. TFS-061).
- onset_breakpoint_Ma: breakpoint where the main-episode slope sets in (fit-based);
  empty if the main segment starts at the model start.
onset_T_in_apaz tells whether the peak temperature lies inside the APAZ +/- APAZ_TOL_C.
onset_robust = same onset class and both onset ages within ROBUST_SPREAD_MYR for every
floor in SENSITIVITY_SIGMAS.

Local fit check (v3.1): the BIC is a global score, so a large misfit confined to a
short stretch of the path can survive it (e.g. TFS-116). After the BIC choice, the
number of breakpoints is increased until, inside the APAZ, the fit departs from the
expected path by no more than MAX_MISFIT_APAZ_C (5 degC); max_misfit_apaz_C is reported.

Changes v2 -> v3: SIGMA_FLOOR_C 4 -> 3 degC (4 degC merged real early heating
phases into the main episode); path-based censoring; two onset columns.

Caveats (state them in the Methods)
-----------------------------------
- The expected path smooths the posterior; sharp inflections in individual paths
  appear rounded. Cross-check key samples against the QTQt maximum-likelihood model.
- Only the part of the path inside the APAZ is constrained by AFT data.
- The fitted corner can overshoot the expected path by a few degrees.

Output (in OUT_DIR):
    <sample>_breakpoint_segments.csv
    <sample>_breakpoint_plot.png / .pdf
    ALL_SAMPLES_breakpoint_segments.csv     (one row per segment)
    ALL_SAMPLES_breakpoint_summary.csv      (one row per sample)
    ALL_SAMPLES_breakpoint_sensitivity.csv  (n breakpoints / onset vs SIGMA_FLOOR_C)
"""
import os
import re
import glob
import csv
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.lines import Line2D
from scipy.optimize import differential_evolution

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
from config import TT_DIR as FOLDER_PATH, OUT
FILE_PATTERN = "*expected Tt file.txt"
OUT_DIR = os.path.join(OUT, "breakpoints")

MAX_BREAKS = 6                # largest number of breakpoints tested
SIGMA_FLOOR_C = 3.0           # temperature resolution floor for the BIC (degC)
SENSITIVITY_SIGMAS = [2.0, 3.0, 4.0, 5.0, 6.0]
ROBUST_SPREAD_MYR = 10.0      # onset counted as robust if it varies <= this across SENSITIVITY_SIGMAS
MAX_MISFIT_APAZ_C = 5.0       # inside the APAZ the fit may not depart from the path by more than this
MIN_SEGMENT_MYR = 10.0        # shortest allowed segment (Myr)
MIN_SLOPE_CHANGE_C_MYR = 0.1  # adjacent segments must differ by at least this (degC/Myr)
RANDOM_SEED = 42

APAZ_LOW_C, APAZ_HIGH_C = 60.0, 120.0   # APAZ ~60-120 degC (as in Fonseca et al.)
APAZ_TOL_C = 5.0              # tolerance for "onset temperature inside the APAZ"
MIN_APAZ_FRACTION = 0.5

GRADIENT_LOW_C_KM = 18.0      # Alexandrino & Hamza (2008)
GRADIENT_MID_C_KM = 25.0      # Jelinek et al. (2014)
GRADIENT_HIGH_C_KM = 30.0     # Hamza et al. (2005)

fm.findSystemFonts(fontpaths=None, fontext="ttf")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Montserrat", "DejaVu Sans", "Arial"]
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["axes.formatter.use_locale"] = False


# ---------------------------------------------------------------------------
# INPUT
# ---------------------------------------------------------------------------
def load_tt_path(path):
    """QTQt Expected Model export -> arrays sorted oldest -> youngest, anchor removed."""
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()
    for ln in lines[1:]:
        parts = ln.split()
        if len(parts) != 4:
            continue
        try:
            rows.append(tuple(float(x) for x in parts))
        except ValueError:
            continue
    rows.sort(key=lambda r: -r[0])
    t, T, lo, hi = (np.array(c) for c in zip(*rows))
    anchor = (t == t.max()) & (T == 0) & (lo == 0) & (hi == 0)
    return t[~anchor], T[~anchor], lo[~anchor], hi[~anchor]


def sample_name_from_path(path):
    stem = os.path.splitext(os.path.basename(path))[0]
    m = re.match(r"(.+?)[ _]?(expected[ _])?Tt[ _]?file$", stem, flags=re.IGNORECASE)
    return m.group(1).rstrip("_ ") if m else stem


# ---------------------------------------------------------------------------
# CONTINUOUS PIECEWISE-LINEAR (HINGE) MODEL
# ---------------------------------------------------------------------------
def design_matrix(t, bps):
    """T = a + b*t + sum_i c_i * max(0, bp_i - t): continuous, slope changes at each bp."""
    cols = [np.ones_like(t), t] + [np.maximum(0.0, bp - t) for bp in bps]
    return np.column_stack(cols)


def fit_fixed_breaks(t, T, bps):
    X = design_matrix(t, bps)
    coef, *_ = np.linalg.lstsq(X, T, rcond=None)
    pred = X @ coef
    return coef, pred, float(np.sum((T - pred) ** 2))


def valid_breaks(bps, t_min, t_max):
    edges = np.concatenate([[t_max], np.sort(bps)[::-1], [t_min]])
    return np.all(-np.diff(edges) >= MIN_SEGMENT_MYR)


def optimise_breaks(t, T, k):
    """Best k breakpoint ages (sorted, oldest first) by least squares."""
    t_min, t_max = float(t.min()), float(t.max())
    if k == 0:
        _, pred, rss = fit_fixed_breaks(t, T, [])
        return np.array([]), rss
    if (t_max - t_min) < (k + 1) * MIN_SEGMENT_MYR:
        return None

    def objective(x):
        if not valid_breaks(x, t_min, t_max):
            return 1e12
        return fit_fixed_breaks(t, T, x)[2]

    bounds = [(t_min + MIN_SEGMENT_MYR, t_max - MIN_SEGMENT_MYR)] * k
    res = differential_evolution(objective, bounds, seed=RANDOM_SEED, tol=1e-8,
                                 maxiter=400, popsize=20, polish=True)
    bps = np.sort(res.x)[::-1]
    if not valid_breaks(bps, t_min, t_max):
        return None
    return bps, fit_fixed_breaks(t, T, bps)[2]


def bic(rss, n, k, sigma):
    rss_eff = max(rss, n * sigma ** 2)
    return n * np.log(rss_eff / n) + (2 * k + 2) * np.log(n)


def segment_slopes(t, pred, bps):
    """Slope (dT/d(age), +ve = cooling forward in time) of each segment of the fit."""
    edges = np.concatenate([[t.max()], bps, [t.min()]])
    Tfit = np.interp(edges, t[::-1], pred[::-1])
    return edges, Tfit, (Tfit[:-1] - Tfit[1:]) / (edges[:-1] - edges[1:])


def prune_small_slope_changes(t, T, bps):
    """Drop breakpoints where adjacent slopes differ by < MIN_SLOPE_CHANGE; refit."""
    bps = np.array(bps)
    while len(bps):
        _, pred, _ = fit_fixed_breaks(t, T, bps)
        _, _, slopes = segment_slopes(t, pred, bps)
        dslope = np.abs(np.diff(slopes))
        i = int(np.argmin(dslope))
        if dslope[i] >= MIN_SLOPE_CHANGE_C_MYR:
            break
        bps = np.delete(bps, i)
    return bps


def fit_all_k(t, T):
    """Least-squares fits for every k (independent of SIGMA_FLOOR_C)."""
    table = []
    for k in range(MAX_BREAKS + 1):
        out = optimise_breaks(t, T, k)
        if out is None:
            break
        table.append(dict(k=k, bps=out[0], rss=out[1]))
    return table


def max_misfit_in_apaz(t, T, bps):
    """Largest |expected path - fit| over points inside the APAZ (+/- tolerance)."""
    _, pred, _ = fit_fixed_breaks(t, T, bps)
    inside = (T >= APAZ_LOW_C - APAZ_TOL_C) & (T <= APAZ_HIGH_C + APAZ_TOL_C)
    return float(np.max(np.abs(T - pred)[inside])) if inside.any() else 0.0


def choose(t, T, table, sigma):
    """BIC choice, then a local check: inside the APAZ the fit must follow the
    expected path within MAX_MISFIT_APAZ_C. The BIC is a global score, so a large
    misfit confined to a short stretch (e.g. a sigmoid-shaped cooling episode near
    the start of the model) does not pay for an extra breakpoint; in that case k is
    increased until the local criterion is met or MAX_BREAKS is reached."""
    n = len(t)
    scores = [bic(r["rss"], n, r["k"], sigma) for r in table]
    i = int(np.argmin(scores))
    while i < len(table) - 1 and max_misfit_in_apaz(t, T, table[i]["bps"]) > MAX_MISFIT_APAZ_C:
        i += 1
    best = table[i]
    bps = prune_small_slope_changes(t, T, best["bps"])
    return bps, best["k"], scores


# ---------------------------------------------------------------------------
# SEGMENTS, RATES, MAIN COOLING
# ---------------------------------------------------------------------------
def drop_inside_apaz(T_start, T_end):
    """Temperature drop of a cooling segment that falls inside the APAZ."""
    hi_ = min(T_start, APAZ_HIGH_C)
    lo_ = max(T_end, APAZ_LOW_C)
    return max(0.0, hi_ - lo_)


def build_segments(t, T, lo, hi, bps):
    _, pred, _ = fit_fixed_breaks(t, T, bps)
    edges, Tfit, slopes = segment_slopes(t, pred, bps)
    segs = []
    for i, rate in enumerate(slopes):
        t_old, t_young = edges[i], edges[i + 1]
        sel = (t <= t_old) & (t >= t_young)
        Ts = T[sel]
        frac = float(np.mean((Ts >= APAZ_LOW_C) & (Ts <= APAZ_HIGH_C))) if sel.any() else 0.0
        cooling = rate > 0
        d_apaz = drop_inside_apaz(Tfit[i], Tfit[i + 1]) if cooling else 0.0
        dT = Tfit[i] - Tfit[i + 1]
        resid = Ts - np.interp(t[sel], [t_young, t_old], [Tfit[i + 1], Tfit[i]])
        r2_den = np.sum((Ts - Ts.mean()) ** 2)
        s = dict(
            t_start=t_old, t_end=t_young, T_start=Tfit[i], T_end=Tfit[i + 1],
            duration_myr=t_old - t_young, delta_T_c=dT, delta_T_in_apaz_c=d_apaz,
            cooling_rate_c_myr=rate,
            rms_misfit_c=float(np.sqrt(np.mean(resid ** 2))) if sel.any() else np.nan,
            r2=float(1 - np.sum(resid ** 2) / r2_den) if r2_den > 0 else np.nan,
            mean_ci_halfwidth_c=float(np.mean((hi[sel] - lo[sel]) / 2)) if sel.any() else np.nan,
            apaz_fraction=frac,
            resolved=bool(frac >= MIN_APAZ_FRACTION or (cooling and d_apaz >= 0.5 * abs(dT))),
            regime="cooling" if cooling else "heating (burial?)",
            main_cooling=False,
        )
        for label, grad in (("low", GRADIENT_HIGH_C_KM), ("mid", GRADIENT_MID_C_KM),
                            ("high", GRADIENT_LOW_C_KM)):
            s[f"exhumation_{label}_m_myr"] = rate / grad * 1000 if cooling else np.nan
        segs.append(s)

    candidates = [s for s in segs if s["delta_T_in_apaz_c"] > 0]
    main = max(candidates, key=lambda s: (s["delta_T_in_apaz_c"], s["cooling_rate_c_myr"])) \
        if candidates else None
    if main:
        main["main_cooling"] = True
    return segs, main


def onset_info(main, t, T):
    """Two onset measures for the main cooling episode.

    peak_T_age_Ma  - age of the maximum temperature of the EXPECTED PATH older than
                     the end of the main episode = start of net cooling. Read from the
                     path itself, so it does not depend on the piecewise fit or on
                     SIGMA_FLOOR_C (same definition as t_in in results_combined_figure.py).
                     Censored if the peak lies at the start of the modelled time range.
    onset_breakpoint_Ma - breakpoint where the main-episode slope sets in (fit-based).
                     Empty if the main segment starts at the model start.
    """
    empty = dict(onset_type="no cooling through APAZ resolved", peak_T_age_Ma=np.nan,
                 peak_T_C=np.nan, onset_min_Ma=np.nan, onset_breakpoint_Ma=np.nan,
                 onset_T_in_apaz=False)
    if main is None:
        return empty
    sel = t >= main["t_end"]
    ipk = int(np.argmax(T[sel]))
    t_pk, T_pk = float(t[sel][ipk]), float(T[sel][ipk])
    censored = t_pk >= t.max() - MIN_SEGMENT_MYR
    bp_at_start = main["t_start"] >= t.max() - 0.5   # main segment is the first one
    in_apaz = (APAZ_LOW_C - APAZ_TOL_C) <= T_pk <= (APAZ_HIGH_C + APAZ_TOL_C)
    return dict(
        onset_type=f"censored (>= {t.max():.0f} Ma)" if censored else "resolved",
        peak_T_age_Ma=np.nan if censored else t_pk,
        peak_T_C=T_pk,
        onset_min_Ma=float(t.max()) if censored else np.nan,
        onset_breakpoint_Ma=np.nan if bp_at_start else main["t_start"],
        onset_T_in_apaz=bool(in_apaz),
    )


# ---------------------------------------------------------------------------
# FIGURE
# ---------------------------------------------------------------------------
def plot_sample(name, t, T, lo, hi, segs, main, onset, bps, out_dir):
    fig, ax = plt.subplots(figsize=(9, 6.5))
    ax.axhspan(APAZ_LOW_C, APAZ_HIGH_C, color="#E8E2D0", alpha=0.6, lw=0, zorder=0,
               label=f"APAZ ({APAZ_LOW_C:.0f}\u2013{APAZ_HIGH_C:.0f} \u00b0C)")
    ax.fill_between(t, lo, hi, color="#4C78A8", alpha=0.18, lw=0, label="95% credible interval")
    ax.plot(t, T, color="#1b1b1b", lw=2, label="Expected model")

    for s in segs:
        color = "#B33951" if s["resolved"] else "#9a9a9a"
        lw = 3.4 if s["main_cooling"] else 2.0
        ax.plot([s["t_start"], s["t_end"]], [s["T_start"], s["T_end"]], color=color, lw=lw,
                ls="-" if s["resolved"] else (0, (4, 2)), zorder=4)
        xm = (s["t_start"] + s["t_end"]) / 2
        ym = min(s["T_start"], s["T_end"]) - 6
        if s["regime"] == "cooling":
            txt = f"{s['cooling_rate_c_myr']:.2f} \u00b0C/Myr\n{s['exhumation_mid_m_myr']:.1f} m/Myr"
        else:
            txt = f"heating\n{s['cooling_rate_c_myr']:.2f} \u00b0C/Myr"
        if not s["resolved"]:
            txt += "\n(outside APAZ)"
        ax.annotate(txt, (xm, ym), ha="center", va="bottom", fontsize=7.5, color=color)

    for i, bp in enumerate(bps):
        Tb = segs[i]["T_end"]
        is_onset = np.isfinite(onset["onset_breakpoint_Ma"]) and np.isclose(bp, onset["onset_breakpoint_Ma"])
        ax.plot(bp, Tb, marker="D" if is_onset else "o", ms=8 if is_onset else 6,
                mfc="#F2B134" if is_onset else "white", mec="#1b1b1b", zorder=6)
        ax.annotate(f"{bp:.0f} Ma", (bp, Tb), xytext=(0, 12), textcoords="offset points",
                    ha="center", fontsize=8, fontweight="bold" if is_onset else "normal")

    if onset["onset_type"] == "resolved":
        # peak paleotemperature of the expected path = start of net cooling
        ax.plot(onset["peak_T_age_Ma"], onset["peak_T_C"], marker="*", ms=14,
                mfc="#F2B134", mec="#1b1b1b", zorder=7)
        ax.annotate(f"peak {onset['peak_T_age_Ma']:.0f} Ma", (onset["peak_T_age_Ma"], onset["peak_T_C"]),
                    xytext=(0, -14), textcoords="offset points", ha="center", va="top",
                    fontsize=8, fontweight="bold", color="#8a5a00")
    elif onset["onset_type"].startswith("censored"):
        T0 = float(T[np.argmax(t)])
        ax.annotate(f"onset \u2265 {onset['onset_min_Ma']:.0f} Ma\n(peak T at model start)",
                    (t.max(), T0), xytext=(10, -16), textcoords="offset points",
                    ha="left", va="top", fontsize=8, fontweight="bold", color="#8a5a00")
        ax.plot(t.max(), T0, marker=">", ms=9, mfc="#F2B134", mec="#1b1b1b", zorder=6)
    elif main is None:
        ax.text(0.02, 0.97, "No cooling through the APAZ resolved by the expected path",
                transform=ax.transAxes, fontsize=9, va="top", color="#8a5a00", fontweight="bold")

    ax.invert_xaxis()
    label_tops = [min(s["T_start"], s["T_end"]) - 6 for s in segs] or [np.nanmin(T)]
    ax.set_ylim(max(np.nanmax(hi), APAZ_HIGH_C) + 5, min(np.nanmin(lo), min(label_tops)) - 22)
    ax.set_xlabel("Time (Ma)", fontsize=12)
    ax.set_ylabel("Temperature (\u00b0C)", fontsize=12)
    ax.set_title(name, fontsize=18, loc="left", fontweight="bold")
    ax.grid(alpha=0.2)

    h, l = ax.get_legend_handles_labels()
    h += [Line2D([0], [0], color="#B33951", lw=2.0),
          Line2D([0], [0], color="#B33951", lw=3.4),
          Line2D([0], [0], color="#9a9a9a", lw=2.0, ls=(0, (4, 2))),
          Line2D([0], [0], marker="o", ls="", mfc="white", mec="#1b1b1b"),
          Line2D([0], [0], marker="D", ls="", mfc="#F2B134", mec="#1b1b1b", ms=8),
          Line2D([0], [0], marker="*", ls="", mfc="#F2B134", mec="#1b1b1b", ms=12),
          Line2D([0], [0], marker=">", ls="", mfc="#F2B134", mec="#1b1b1b", ms=8)]
    l += ["Piecewise fit (resolved)", "Main cooling episode", "Piecewise fit (outside APAZ)",
          "Breakpoint", "Onset of main-episode slope", "Peak paleotemperature (onset of net cooling)",
          "Onset censored (peak T at model start)"]
    ax.legend(h, l, loc="upper center", bbox_to_anchor=(0.5, -0.11), ncol=3,
              fontsize=8, frameon=False)

    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, f"{name}_breakpoint_plot.png"), dpi=300, bbox_inches="tight")
    fig.savefig(os.path.join(out_dir, f"{name}_breakpoint_plot.pdf"), bbox_inches="tight")
    plt.show()
    plt.close(fig)


# ---------------------------------------------------------------------------
# OUTPUT TABLES
# ---------------------------------------------------------------------------
SEG_KEYS = ["sample", "segment", "t_start", "t_end", "T_start", "T_end", "duration_myr",
            "delta_T_c", "delta_T_in_apaz_c", "cooling_rate_c_myr", "rms_misfit_c", "r2",
            "mean_ci_halfwidth_c", "apaz_fraction", "resolved", "regime", "main_cooling",
            "exhumation_low_m_myr", "exhumation_mid_m_myr", "exhumation_high_m_myr"]
SUM_KEYS = ["sample", "t_path_start_Ma", "max_T_path_C", "n_breakpoints", "breakpoints_Ma",
            "onset_type", "peak_T_age_Ma", "onset_min_Ma", "peak_T_C", "onset_T_in_apaz",
            "onset_breakpoint_Ma", "main_cooling_end_Ma", "main_cooling_rate_c_myr",
            "main_delta_T_in_apaz_c",
            "main_exhumation_low_m_myr", "main_exhumation_mid_m_myr", "main_exhumation_high_m_myr",
            "onset_robust", "breakpoint_onset_over_sigmas", "max_misfit_apaz_C", "best_k_by_BIC", "best_k_at_max", "breakpoint_at_young_bound"] + \
           [f"BIC_k{k}" for k in range(MAX_BREAKS + 1)]


def fmt(v):
    if isinstance(v, (bool, np.bool_)):
        return str(bool(v))
    if isinstance(v, (float, np.floating)):
        return "" if np.isnan(v) else f"{v:.4f}"
    return v


def write_csv(path, keys, rows):
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in rows:
            w.writerow({k: fmt(r.get(k)) for k in keys})


# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------
def process(path, out_dir, plot=True):
    name = sample_name_from_path(path)
    t, T, lo, hi = load_tt_path(path)
    table = fit_all_k(t, T)

    # sensitivity to the resolution floor (fits reused, only the BIC changes)
    sens = {"sample": name}
    sens_types, sens_bp, sens_pk = [], [], []
    for sg in SENSITIVITY_SIGMAS:
        b, _, _ = choose(t, T, table, sg)
        sg_segs, sg_main = build_segments(t, T, lo, hi, b)
        sg_on = onset_info(sg_main, t, T)
        sens_types.append(sg_on["onset_type"])
        sens_bp.append(sg_on["onset_breakpoint_Ma"])
        sens_pk.append(sg_on["peak_T_age_Ma"])
        sens[f"n_bp_sigma{sg:g}"] = len(b)
        sens[f"onset_sigma{sg:g}"] = (
            ("peak " + (f"{sg_on['peak_T_age_Ma']:.0f}" if np.isfinite(sg_on["peak_T_age_Ma"])
                        else sg_on["onset_type"].split(" (")[0]))
            + (f" | bp {sg_on['onset_breakpoint_Ma']:.0f}" if np.isfinite(sg_on["onset_breakpoint_Ma"]) else ""))

    def spread_of(vals):
        a = np.array(vals, dtype=float)
        if np.all(np.isnan(a)):
            return 0.0
        if np.any(np.isnan(a)):
            return np.nan            # defined for some floors only -> not robust
        return float(a.max() - a.min())

    sp_bp, sp_pk = spread_of(sens_bp), spread_of(sens_pk)
    robust = (len(set(sens_types)) == 1
              and np.isfinite(sp_bp) and sp_bp <= ROBUST_SPREAD_MYR
              and np.isfinite(sp_pk) and sp_pk <= ROBUST_SPREAD_MYR)
    sens["onset_robust"] = robust
    sens["breakpoint_onset_spread_Myr"] = sp_bp
    sens["peak_age_spread_Myr"] = sp_pk

    bps, best_k, scores = choose(t, T, table, SIGMA_FLOOR_C)
    segs, main = build_segments(t, T, lo, hi, bps)
    onset = onset_info(main, t, T)

    seg_rows = [{"sample": name, "segment": i + 1, **s} for i, s in enumerate(segs)]
    write_csv(os.path.join(out_dir, f"{name}_breakpoint_segments.csv"), SEG_KEYS, seg_rows)
    if plot:
        plot_sample(name, t, T, lo, hi, segs, main, onset, bps, out_dir)

    m = main or {}
    summary = dict(
        sample=name, t_path_start_Ma=float(t.max()), max_T_path_C=float(np.nanmax(T)),
        n_breakpoints=len(bps), breakpoints_Ma="; ".join(f"{b:.1f}" for b in bps),
        **onset,
        main_cooling_end_Ma=m.get("t_end", np.nan),
        main_cooling_rate_c_myr=m.get("cooling_rate_c_myr", np.nan),
        main_delta_T_in_apaz_c=m.get("delta_T_in_apaz_c", np.nan),
        main_exhumation_low_m_myr=m.get("exhumation_low_m_myr", np.nan),
        main_exhumation_mid_m_myr=m.get("exhumation_mid_m_myr", np.nan),
        main_exhumation_high_m_myr=m.get("exhumation_high_m_myr", np.nan),
        onset_robust=robust,
        breakpoint_onset_over_sigmas="; ".join(f"{a:.0f}" if np.isfinite(a) else "-" for a in sens_bp),
        max_misfit_apaz_C=max_misfit_in_apaz(t, T, bps),
        best_k_by_BIC=best_k, best_k_at_max=best_k == table[-1]["k"],
        breakpoint_at_young_bound=bool(np.any(np.isclose(bps, t.min() + MIN_SEGMENT_MYR, atol=0.5))),
        **{f"BIC_k{r['k']}": sc for r, sc in zip(table, scores)},
    )
    return seg_rows, summary, sens


def main(folder=FOLDER_PATH, out_dir=OUT_DIR, plot=True):
    os.makedirs(out_dir, exist_ok=True)
    files = sorted(glob.glob(os.path.join(folder, FILE_PATTERN)))
    if not files:
        print(f"No files matching '{FILE_PATTERN}' in: {folder}")
        return
    all_segs, all_sum, all_sens = [], [], []
    for path in files:
        name = sample_name_from_path(path)
        print(f"Processing {name} ...")
        try:
            segs, summ, sens = process(path, out_dir, plot=plot)
            all_segs += segs
            all_sum.append(summ)
            all_sens.append(sens)
            on = (f"peak T at {summ['peak_T_age_Ma']:.0f} Ma" if summ["onset_type"] == "resolved"
                  else summ["onset_type"])
            bp = summ["onset_breakpoint_Ma"]
            on += f", main slope from {bp:.0f} Ma" if np.isfinite(bp) else ""
            print(f"  breakpoints: {summ['breakpoints_Ma'] or 'none'} | onset: {on}")
        except Exception as e:
            print(f"  -> ERROR ({name}): {e}")
    if all_segs:
        write_csv(os.path.join(out_dir, "ALL_SAMPLES_breakpoint_segments.csv"), SEG_KEYS, all_segs)
        write_csv(os.path.join(out_dir, "ALL_SAMPLES_breakpoint_summary.csv"), SUM_KEYS, all_sum)
        sens_keys = ["sample"] + [f"{p}_sigma{sg:g}" for sg in SENSITIVITY_SIGMAS
                                  for p in ("n_bp", "onset")] + \
                    ["onset_robust", "breakpoint_onset_spread_Myr", "peak_age_spread_Myr"]
        write_csv(os.path.join(out_dir, "ALL_SAMPLES_breakpoint_sensitivity.csv"), sens_keys, all_sens)
    print(f"\nDone. {len(files)} file(s). Output: {out_dir}")


if __name__ == "__main__":
    main()


## 6. Cooling histories, step 2: posterior cooling rates (Table 3; Supplementary Fig. S5)

Cooling rate over the main cooling episode computed on 10⁴ posterior thermal histories of each QTQt chain. Run after step 1.

Script: `figS5b_posterior_rates.py`

In [ ]:
%%writefile figS5b_posterior_rates.py
"""
Step 2 of 2 (run after 1_breakpoints.py).
QTQt posterior cooling rates - rates computed on every sampled thermal history
=============================================================================
The expected path is the mean of the QTQt posterior. Its slope gives the mean
rate but no honest uncertainty, and averaging smooths out short pulses. QTQt
writes every post-burn-in model of the MCMC chain to "<run name>.txt"
(lines after "CHAIN": iteration, log-likelihood, log-posterior, N, then N+1
(time, temperature) pairs, the last one being the present-day point). This
script reads those chains and computes the rates on each model, giving a full
posterior distribution for:

  1. the MAIN COOLING EPISODE identified by 1_breakpoints.py:
     chord rate between the episode's start and end ages on every model;
  2. the age of PEAK PALEOTEMPERATURE before the end of the main episode
     (start of net cooling), per model, with the fraction of models whose peak
     lies at the start of the modelled time range (censored);
  3. optional FIXED TIME WINDOWS (150, 100, 50, 20, 0 Ma; supplementary only).

Note: the mean of the per-model chord rates equals the chord rate of the expected
path over the same window (the operations are linear) - a built-in check. The new
information is the spread (credible interval) and the probability of cooling.

Choosing the chain file
-----------------------
Run names differ between samples ("No constraints2", "Run3", "tentativa 3",
"2", ...), and one folder holds files named after another sample. The script
therefore does NOT rely on names: it reads every chain in the sample folder and
keeps the runs whose posterior mean reproduces the expected Tt file used in the
other scripts within MATCH_WARN_C, and among those uses the MOST RECENT run (the
final one). If none matches, the closest run is used and flagged. The chosen file and the match quality
are written to the summary - check them. A mismatch > MATCH_WARN_C is flagged.

Interpretation (state in the Methods)
-------------------------------------
Rates are meaningful only where the paths lie inside the APAZ (~60-120 degC);
the fixed-window table carries the 'resolved' flag from the expected path.

Inputs
------
CHAIN_ROOT/<sample>/*.txt          QTQt run outputs (one folder per sample)
TT_FOLDER/<sample> expected Tt file.txt
SEGMENTS_CSV                       ALL_SAMPLES_breakpoint_segments.csv (from 1_breakpoints.py)

Outputs (OUT_DIR)
-----------------
ALL_SAMPLES_posterior_main_episode.csv   one row per sample
ALL_SAMPLES_posterior_windows.csv        one row per sample x window
<sample>_posterior_rate.png / .pdf       posterior t-T density + rate histogram
ALL_SAMPLES_main_rate_forest.png / .pdf  median and 95% CI of the main-episode rate
"""
import os
import re
import glob
import csv
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
# QTQt run folders, in priority order: the reruns in "Modelling - FINAL" win over
# the older runs of the same sample in "projected round 2"
from config import QTQT_DIRS as CHAIN_ROOTS, TT_DIR, OUT
CHAIN_ROOT = CHAIN_ROOTS[-1]   # kept for messages
SEGMENTS_CSV = os.path.join(OUT, "breakpoints", "ALL_SAMPLES_breakpoint_segments.csv")
OUT_DIR = os.path.join(OUT, "posterior_rates")

THIN = 10                 # keep every THIN-th model (100,000 -> 10,000)
THIN_MATCH = 50           # thinning used only to identify the right chain file
MATCH_WARN_C = 2.0        # flag if the chosen chain's mean departs from the expected file by more
GRID_STEP_MYR = 1.0

FIXED_BREAKPOINTS_MA = [150, 100, 50, 20, 0]
APAZ_LOW_C, APAZ_HIGH_C = 60.0, 120.0   # APAZ ~60-120 degC (as in Fonseca et al.)
MIN_APAZ_FRACTION = 0.5
MIN_SEGMENT_MYR = 10.0    # peak within this of the model start = censored

GRADIENT_LOW_C_KM = 18.0
GRADIENT_MID_C_KM = 25.0
GRADIENT_HIGH_C_KM = 30.0

PCTS = [2.5, 16, 50, 84, 97.5]

fm.findSystemFonts(fontpaths=None, fontext="ttf")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Montserrat", "DejaVu Sans", "Arial"]
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["axes.formatter.use_locale"] = False


# ---------------------------------------------------------------------------
# READING
# ---------------------------------------------------------------------------
EXCLUDE = ("_pred", "_oldesttrack", "_tto_fix", "qtqt pc", "documentation")


def is_chain_file(path):
    name = os.path.basename(path).lower()
    if any(k in name for k in EXCLUDE):
        return False
    try:
        with open(path, "r", errors="ignore") as f:
            for _ in range(80):
                ln = f.readline()
                if not ln:
                    break
                if ln.startswith("CHAIN"):
                    return True
    except OSError:
        return False
    return False


def read_chain(path, thin=1):
    """Return list of (t, T) arrays sorted by time (young -> old)."""
    models = []
    with open(path, "r", errors="ignore") as fh:
        on, i = False, 0
        for ln in fh:
            if ln.startswith("CHAIN END"):
                break
            if ln.startswith("CHAIN"):
                on = True
                continue
            if not on:
                continue
            p = ln.split()
            try:
                n = int(p[3])
            except (IndexError, ValueError):
                break
            i += 1
            if (i - 1) % thin:
                continue
            v = np.array(p[4:], dtype=float)
            if len(v) != 2 * (n + 1):
                break
            t, T = v[0::2], v[1::2]
            o = np.argsort(t)
            models.append((t[o], T[o]))
    return models


def read_expected(sample):
    path = os.path.join(TT_FOLDER, f"{sample} expected Tt file.txt")
    rows = []
    with open(path, "r", errors="ignore") as f:
        for ln in f.readlines()[1:]:
            p = ln.split()
            if len(p) != 4:
                continue
            try:
                rows.append([float(x) for x in p])
            except ValueError:
                pass
    a = np.array(rows)
    a = a[~((a[:, 1] == 0) & (a[:, 2] == 0) & (a[:, 3] == 0))]   # drop box anchor row
    a = a[np.argsort(a[:, 0])]
    return a[:, 0], a[:, 1]


def models_on_grid(models, grid):
    return np.array([np.interp(grid, t, T, left=np.nan, right=np.nan) for t, T in models])


def choose_chain(sample, grid, T_exp):
    cands = []
    for rank, root in enumerate(CHAIN_ROOTS):
        cands += [(rank, p) for p in glob.glob(os.path.join(root, sample, "*.txt")) if is_chain_file(p)]
    scored, rank_of = [], {}
    for rank, p in cands:
        rank_of[p] = rank
        m = read_chain(p, thin=THIN_MATCH)
        if len(m) < 50:
            scored.append((np.inf, p))
            continue
        mean = np.nanmean(models_on_grid(m, grid), axis=0)
        scored.append((float(np.nanmean(np.abs(mean - T_exp))), p))
    if not scored:
        return None, np.inf, []
    scored.sort()
    # several runs of the same sample usually reproduce the expected path almost
    # equally well; among those within MATCH_WARN_C, keep the run from the
    # highest-priority folder (Modelling - FINAL first), then the most recent one
    ok = [(d, p) for d, p in scored if d <= MATCH_WARN_C]
    if ok:
        d, p = min(ok, key=lambda dp: (rank_of[dp[1]], -os.path.getmtime(dp[1])))
        others = [(dd, pp) for dd, pp in scored if pp != p]
        return p, d, [(d, p)] + others
    return scored[0][1], scored[0][0], scored


# ---------------------------------------------------------------------------
# RATES
# ---------------------------------------------------------------------------
def chord_rates(Tg, grid, t_old, t_young):
    """Per-model chord rate (degC/Myr, +ve = cooling) between two ages."""
    T_old = np.array([np.interp(t_old, grid, row, left=np.nan, right=np.nan) for row in Tg])
    T_young = np.array([np.interp(t_young, grid, row, left=np.nan, right=np.nan) for row in Tg])
    r = (T_old - T_young) / (t_old - t_young)
    return r[np.isfinite(r)]


def peak_ages(Tg, grid, t_end, t_max):
    sel = (grid >= t_end) & (grid <= t_max)
    sub = Tg[:, sel]
    ok = np.all(np.isfinite(sub), axis=1)
    idx = np.argmax(sub[ok], axis=1)
    return grid[sel][idx]


def pct(x):
    return np.percentile(x, PCTS) if len(x) else np.full(len(PCTS), np.nan)


def load_main_windows():
    out = {}
    with open(SEGMENTS_CSV, newline="") as f:
        for r in csv.DictReader(f):
            if r["main_cooling"].strip().lower() == "true":
                out[r["sample"]] = (float(r["t_start"]), float(r["t_end"]))
    return out


# ---------------------------------------------------------------------------
# FIGURES
# ---------------------------------------------------------------------------
def plot_sample(name, grid, Tg, t_exp, T_exp, window, rates, out_dir):
    fig, (ax, axh) = plt.subplots(1, 2, figsize=(12, 5.2), gridspec_kw={"width_ratios": [2.2, 1]})

    # posterior density of paths, normalised per time column (as in QTQt)
    tbins = np.arange(0, 141, 2.0)
    H = np.zeros((len(tbins) - 1, len(grid)))
    for j in range(len(grid)):
        col = Tg[:, j]
        col = col[np.isfinite(col)]
        if len(col):
            h, _ = np.histogram(col, bins=tbins)
            H[:, j] = h / h.max() if h.max() else h
    ax.imshow(H, origin="lower", aspect="auto", cmap="Blues",
              extent=[grid[0], grid[-1], tbins[0], tbins[-1]], vmin=0, vmax=1, zorder=0)
    ax.axhspan(APAZ_LOW_C, APAZ_HIGH_C, color="#E8E2D0", alpha=0.35, lw=0, zorder=1)
    ax.plot(t_exp, T_exp, color="#1b1b1b", lw=2, zorder=3, label="Expected model")
    if window:
        ax.axvspan(window[1], window[0], color="#B33951", alpha=0.12, lw=0, zorder=2,
                   label="Main cooling episode")
    ax.set_xlim(grid[-1], grid[0])
    ax.set_ylim(140, 0)
    ax.set_xlabel("Time (Ma)", fontsize=11)
    ax.set_ylabel("Temperature (\u00b0C)", fontsize=11)
    ax.set_title(name, fontsize=16, loc="left", fontweight="bold")
    ax.legend(loc="lower left", fontsize=8, frameon=True)

    if len(rates):
        lo, p16, med, p84, hi = pct(rates)
        rng = np.percentile(rates, [0.5, 99.5])
        counts, _, _ = axh.hist(rates, bins=np.linspace(rng[0], rng[1], 50), color="#B33951", alpha=0.75)
        axh.set_ylim(0, counts.max() * 1.45)   # headroom for the statistics box
        for v, ls in ((med, "-"), (lo, "--"), (hi, "--")):
            axh.axvline(v, color="#1b1b1b", lw=1.2, ls=ls)
        axh.axvline(0, color="#999999", lw=0.8)
        axh.set_title(f"Main-episode rate\n{window[0]:.0f}\u2013{window[1]:.0f} Ma", fontsize=10, loc="left")
        axh.text(0.97, 0.95,
                 f"median {med:.2f} \u00b0C/Myr\n95%: {lo:.2f}\u2013{hi:.2f}\n"
                 f"P(cooling) = {np.mean(rates > 0):.2f}\n"
                 f"{med / GRADIENT_MID_C_KM * 1000:.0f} m/Myr (25 \u00b0C/km)",
                 transform=axh.transAxes, ha="right", va="top", fontsize=8.5,
                 bbox=dict(facecolor="white", edgecolor="none", alpha=0.85))
    else:
        axh.text(0.5, 0.5, "No main cooling episode\nresolved in the APAZ", ha="center",
                 va="center", transform=axh.transAxes, fontsize=10, color="#8a5a00")
    axh.set_xlabel("Cooling rate (\u00b0C/Myr)", fontsize=10)
    axh.set_yticks([])
    for s in ("top", "right", "left"):
        axh.spines[s].set_visible(False)

    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, f"{name}_posterior_rate.png"), dpi=300, bbox_inches="tight")
    fig.savefig(os.path.join(out_dir, f"{name}_posterior_rate.pdf"), bbox_inches="tight")
    plt.show()
    plt.close(fig)


def plot_forest(rows, out_dir):
    rows = [r for r in rows if np.isfinite(r.get("rate_median", np.nan))]
    if not rows:
        return
    rows = sorted(rows, key=lambda r: r["rate_median"])
    fig, ax = plt.subplots(figsize=(7, 0.32 * len(rows) + 1.2))
    y = np.arange(len(rows))
    for i, r in enumerate(rows):
        ax.plot([r["rate_p2.5"], r["rate_p97.5"]], [i, i], color="#B33951", lw=1.2)
        ax.plot([r["rate_p16"], r["rate_p84"]], [i, i], color="#B33951", lw=4, alpha=0.6)
        ax.plot(r["rate_median"], i, "o", color="#1b1b1b", ms=4)
    ax.set_yticks(y)
    ax.set_yticklabels([f"{r['sample']} ({r['window_start_Ma']:.0f}\u2013{r['window_end_Ma']:.0f} Ma)" for r in rows],
                       fontsize=8)
    ax.axvline(0, color="#999999", lw=0.8)
    ax.set_xlabel("Main-episode cooling rate (\u00b0C/Myr)  \u2014  median, 68% and 95% credible intervals",
                  fontsize=9)
    ax.grid(axis="x", alpha=0.2)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "ALL_SAMPLES_main_rate_forest.png"), dpi=300, bbox_inches="tight")
    fig.savefig(os.path.join(out_dir, "ALL_SAMPLES_main_rate_forest.pdf"), bbox_inches="tight")
    plt.show()
    plt.close(fig)


# ---------------------------------------------------------------------------
# OUTPUT
# ---------------------------------------------------------------------------
def fmt(v):
    if isinstance(v, (bool, np.bool_)):
        return str(bool(v))
    if isinstance(v, (float, np.floating)):
        return "" if np.isnan(v) else f"{v:.4f}"
    return v


def write_csv(path, keys, rows):
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in rows:
            w.writerow({k: fmt(r.get(k, "")) for k in keys})


MAIN_KEYS = ["sample", "chain_file", "chain_data_file", "chain_data_file_matches_sample", "n_models_used", "match_mean_abs_diff_C", "match_warning",
             "window_start_Ma", "window_end_Ma", "rate_expected_path",
             "rate_p2.5", "rate_p16", "rate_median", "rate_p84", "rate_p97.5", "P_cooling",
             "exh_mid_median_m_myr", "exh_mid_p2.5_m_myr", "exh_mid_p97.5_m_myr",
             "exh_low_median_m_myr", "exh_high_median_m_myr",
             "peak_age_p2.5", "peak_age_median", "peak_age_p97.5", "frac_peak_at_model_start",
             "other_chains_tested"]
WIN_KEYS = ["sample", "t_start", "t_end", "resolved_expected_path", "rate_expected_path",
            "rate_p2.5", "rate_median", "rate_p97.5", "P_cooling", "exh_mid_median_m_myr"]


# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------
def process(sample, windows_main, out_dir, plot=True):
    t_exp, T_exp = read_expected(sample)
    t_max = float(t_exp.max())
    grid = np.arange(0.0, np.floor(t_max) + GRID_STEP_MYR / 2, GRID_STEP_MYR)
    T_exp_g = np.interp(grid, t_exp, T_exp)

    path, diff, scored = choose_chain(sample, grid, T_exp_g)
    if path is None:
        raise FileNotFoundError(f"no QTQt chain file for {sample} in any of {CHAIN_ROOTS}")
    models = read_chain(path, thin=THIN)
    # the chain header (line 3) records the data file QTQt actually read: it must
    # belong to this sample (a run can be saved in the wrong folder / with the wrong data)
    with open(path, "r", errors="ignore") as fh:
        hdr = [fh.readline() for _ in range(3)][2]
    num = re.search(r"(\d+)$", sample).group(1)
    data_file_ok = bool(re.search(r"TFS[ _-]?0*" + str(int(num)) + r"(?!\d)", os.path.basename(hdr.strip().rsplit(" ", 1)[0]), re.I))
    Tg = models_on_grid(models, grid)

    row = dict(sample=sample, chain_file=os.path.basename(path), n_models_used=len(models),
               match_mean_abs_diff_C=diff, match_warning=diff > MATCH_WARN_C,
               chain_data_file=os.path.basename(hdr.strip().rsplit(" ", 1)[0]),
               chain_data_file_matches_sample=data_file_ok,
               other_chains_tested="; ".join(f"{os.path.basename(p)} ({d:.2f})" for d, p in scored[1:]))

    window = windows_main.get(sample)
    rates = np.array([])
    if window:
        t_old, t_young = window
        rates = chord_rates(Tg, grid, t_old, t_young)
        lo, p16, med, p84, hi = pct(rates)
        e_old, e_young = np.interp([t_old, t_young], t_exp, T_exp)
        pk = peak_ages(Tg, grid, t_young, t_max)
        row.update({
            "window_start_Ma": t_old, "window_end_Ma": t_young,
            "rate_expected_path": (e_old - e_young) / (t_old - t_young),
            "rate_p2.5": lo, "rate_p16": p16, "rate_median": med, "rate_p84": p84, "rate_p97.5": hi,
            "P_cooling": float(np.mean(rates > 0)),
            "exh_mid_median_m_myr": med / GRADIENT_MID_C_KM * 1000,
            "exh_mid_p2.5_m_myr": lo / GRADIENT_MID_C_KM * 1000,
            "exh_mid_p97.5_m_myr": hi / GRADIENT_MID_C_KM * 1000,
            "exh_low_median_m_myr": med / GRADIENT_HIGH_C_KM * 1000,
            "exh_high_median_m_myr": med / GRADIENT_LOW_C_KM * 1000,
            "peak_age_p2.5": float(np.percentile(pk, 2.5)) if len(pk) else np.nan,
            "peak_age_median": float(np.median(pk)) if len(pk) else np.nan,
            "peak_age_p97.5": float(np.percentile(pk, 97.5)) if len(pk) else np.nan,
            "frac_peak_at_model_start": float(np.mean(pk >= t_max - MIN_SEGMENT_MYR)) if len(pk) else np.nan,
        })

    win_rows = []
    edges = [t_max] + [b for b in FIXED_BREAKPOINTS_MA if b < t_max]
    for t_old, t_young in zip(edges[:-1], edges[1:]):
        r = chord_rates(Tg, grid, t_old, t_young)
        sel = (t_exp <= t_old) & (t_exp >= t_young)
        frac = np.mean((T_exp[sel] >= APAZ_LOW_C) & (T_exp[sel] <= APAZ_HIGH_C)) if sel.any() else 0.0
        e_old, e_young = np.interp([t_old, t_young], t_exp, T_exp)
        lo, _, med, _, hi = pct(r)
        win_rows.append(dict(sample=sample, t_start=t_old, t_end=t_young,
                             resolved_expected_path=bool(frac >= MIN_APAZ_FRACTION),
                             rate_expected_path=(e_old - e_young) / (t_old - t_young),
                             **{"rate_p2.5": lo, "rate_median": med, "rate_p97.5": hi},
                             P_cooling=float(np.mean(r > 0)) if len(r) else np.nan,
                             exh_mid_median_m_myr=med / GRADIENT_MID_C_KM * 1000))

    if plot:
        plot_sample(sample, grid, Tg, t_exp, T_exp, window, rates, out_dir)
    return row, win_rows


def main(plot=True):
    os.makedirs(OUT_DIR, exist_ok=True)
    windows_main = load_main_windows()
    samples = sorted(os.path.basename(p) for p in glob.glob(os.path.join(TT_FOLDER, "* expected Tt file.txt")))
    samples = [re.sub(r" expected Tt file\.txt$", "", s) for s in samples]
    rows, win_rows = [], []
    for s in samples:
        print(f"Processing {s} ...")
        try:
            row, wr = process(s, windows_main, OUT_DIR, plot=plot)
            rows.append(row)
            win_rows += wr
            msg = f"  chain: {row['chain_file']} (mean |diff| {row['match_mean_abs_diff_C']:.2f} degC)"
            if not row["chain_data_file_matches_sample"]:
                msg += f"  <-- CHECK: chain was run on data file {row['chain_data_file']}"
            if row["match_warning"]:
                msg += "  <-- CHECK: chain does not reproduce the expected file"
            print(msg)
            if np.isfinite(row.get("rate_median", np.nan)):
                print(f"  main episode {row['window_start_Ma']:.0f}-{row['window_end_Ma']:.0f} Ma: "
                      f"{row['rate_median']:.2f} [{row['rate_p2.5']:.2f}-{row['rate_p97.5']:.2f}] degC/Myr")
        except Exception as e:
            print(f"  -> ERROR ({s}): {e}")
    write_csv(os.path.join(OUT_DIR, "ALL_SAMPLES_posterior_main_episode.csv"), MAIN_KEYS, rows)
    write_csv(os.path.join(OUT_DIR, "ALL_SAMPLES_posterior_windows.csv"), WIN_KEYS, win_rows)
    if plot:
        plot_forest(rows, OUT_DIR)
    print(f"\nDone. Output: {OUT_DIR}")


if __name__ == "__main__":
    main()


## 7. Half-boomerang plot (Fig. 6)

AFT central age versus c-axis projected (a) and measured (b) mean track length (Green, 1986).

Script: `fig06_half_boomerang.py`

In [ ]:
%%writefile fig06_half_boomerang.py
"""
4_boomerang_plots.py  --  AFT central age vs mean track length ("boomerang" plot)
================================================================================

Green et al. (1986) boomerang plot for the AFT samples of this study, drawn in
two panels:

  (a) This study: central age (+/-1 sigma) vs c-axis projected MTL.
      The projected MTL is read from the QTQt prediction file (_pred.txt,
      line 3). It is the value QTQt computed and fitted, so it matches the
      thermal models exactly (Ketcham et al. 2007 projection).
  (b) Comparison with the literature, using MEASURED (non-projected) MTL for
      everyone. Most published regional data report only measured MTL, so this
      is the only like-for-like comparison. This study's measured MTL +/- SD
      comes from the 'Length and angle' sheet of each spreadsheet.

Only samples with enough confined tracks are plotted (>= MIN_LENGTHS). Samples
listed in EXCLUDE are always left out (TFS-021: the length data in its
spreadsheet belong to TFS-082). Samples with fewer than ROBUST_LENGTHS tracks
get a thin dashed border (the convention for non-robust samples in the other
figures).

Inputs
------
  data workbook sheets 'Grain counts', 'Samples' and 'Track lengths'
  MODEL_ROOTS   QTQt model folders, in priority order. In each one, the
                sub-folder whose name holds the sample number is searched for
                *_pred.txt. The run is used only if its observed age matches
                the spreadsheet central age (guards against runs that read
                another sample's file).
  LIT_TSV       literature compilation (literature_AFT_data.tsv)

Outputs (OUT_DIR)
-----------------
  boomerang_plot.pdf / .png
  boomerang_data.csv   values plotted, with the source of each projected MTL
"""

import glob
import os
import re

import numpy as np
import openpyxl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.formatter.use_locale"] = False

# ---------------------------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------------------------
from config import sample_names, read_sample, QTQT_DIRS, LIT_TSV, OUT
MODEL_ROOTS = QTQT_DIRS
OUT_DIR = OUT

ZETA, LAMBDA = 266.85, 1.55125e-10
MIN_LENGTHS = 50          # samples below this are not plotted
ROBUST_LENGTHS = 100      # below this: thin dashed border (non-robust)
EXCLUDE = ["TFS-021"]     # length sheet of TFS-021 holds TFS-082 data
AGE_MATCH_TOL_MA = 0.6    # pred-file observed age vs spreadsheet central age
LABEL_SAMPLES = True

# literature: only samples inside this box (lon_min, lon_max, lat_min, lat_max)
LIT_BBOX = (-40.6, -35.5, -12.5, -8.0)   # study area + ~1 degree
# kriging-map extent (UTM 24S 175494-984805 E, 8285374-9420968 N), if preferred:
# LIT_BBOX = (-42.02, -34.49, -15.49, -5.22)

# Tectonic province of each sample (position relative to the SFC / Borborema
# Province boundary on the geological map; same list in every script)
SFC_SAMPLES = {1, 5, 9, 10, 16, 18, 21, 101, 104, 106, 108, 112, 114, 116, 120}   # all others: Borborema
DOMAIN_COLORS = {                       # SFC red / Borborema blue (as in the kriging map)
    "Borborema Province": "#3f6f8f",
    "São Francisco Craton": "#a6423a",
}
DOMAIN_ORDER = ["São Francisco Craton", "Borborema Province"]
# side of the Recôncavo-Tucano-Jatobá rift (marker shape): west = TFS-082 + TFS-101..120
WEST_OF_TUCANO = {82, 101, 104, 106, 108, 112, 114, 116, 120}
SIDE_MARKERS = {"west": "s", "east": "o"}
SIDE_LABELS = {"west": "W of Tucano Basin", "east": "E of Tucano Basin"}
# literature subdomain column -> province (literature points take the province colour)
LIT_PROVINCE = {"sao francisco": "São Francisco Craton", "borborema": "Borborema Province"}
# one shape per reference (open = outline only; "+", "x", "*" are line markers)
LIT_MARKERS = ["^", "+", "D", "*", "x", "v", "p", "h", "o", "s"]
LINE_MARKERS = {"+", "x", "*"}
LIT_COLOR = "#8c8c8c"


def classify_domain(num):
    return "São Francisco Craton" if num in SFC_SAMPLES else "Borborema Province"


# ---------------------------------------------------------------------------
# READING
# ---------------------------------------------------------------------------
def read_sheet(sample):
    """Ns, Ni, rho_d, Nd and confined track lengths of one sample (data workbook)."""
    return read_sample(sample)


def central_age(Ns, Ni, rhod, Nd):
    m, y = Ns + Ni, Ns / (Ns + Ni)
    sig, w = 0.6, np.ones_like(Ns)
    for _ in range(500):
        eta = np.sum(w * y) / np.sum(w)
        w = m / (eta * (1 - eta) + (m - 1) * eta ** 2 * (1 - eta) ** 2 * sig ** 2)
        new = sig * np.sqrt(np.sum(w ** 2 * (y - eta) ** 2) / np.sum(w))
        if abs(new - sig) < 1e-9:
            break
        sig = new
    t = np.log1p(LAMBDA * ZETA * rhod * 0.5 * eta / (1 - eta)) / LAMBDA / 1e6
    rel = np.sqrt((1 / np.sqrt(np.sum(w)) / (eta * (1 - eta))) ** 2 + 1 / Nd)
    return t, t * rel


def projected_mtl(num, age):
    """Observed (projected) MTL from the first matching QTQt _pred.txt."""
    for root in MODEL_ROOTS:
        if not os.path.isdir(root):
            continue
        dirs = [d for d in glob.glob(os.path.join(root, "*"))
                if os.path.isdir(d) and re.search(r"0*%d(?!\d)" % num, os.path.basename(d))]
        files = sorted((p for d in dirs for p in glob.glob(os.path.join(d, "*_pred.txt"))),
                       key=os.path.getmtime, reverse=True)
        for p in files:
            with open(p, errors="ignore") as fh:
                line = [fh.readline() for _ in range(3)][2].split()
            obs_age, obs_mtl = float(line[1]), float(line[2])
            if abs(obs_age - age) <= AGE_MATCH_TOL_MA:
                return obs_mtl, p
    return np.nan, ""


def read_literature(path):
    if not os.path.isfile(path):
        print(f"[warning] literature file not found: {path} (panel b without literature)")
        return []
    fl = lambda s: float(str(s).replace(",", ".")) if str(s).strip() not in ("", "-") else np.nan
    out = []
    with open(path, encoding="utf-8", errors="ignore") as fh:
        hdr = fh.readline().rstrip("\n").split("\t")
        ix = {h: i for i, h in enumerate(hdr)}
        for line in fh:
            c = line.rstrip("\n").split("\t")
            if len(c) < len(hdr) - 1:
                continue
            x, y, age, mtl = (fl(c[ix["coord_X"]]), fl(c[ix["coord_Y"]]),
                              fl(c[ix["AFT_age"]]), fl(c[ix["MTL"]]))
            lo, hi, la, lb = LIT_BBOX
            if np.isnan(mtl) or np.isnan(age) or not (lo <= x <= hi and la <= y <= lb):
                continue
            prov = LIT_PROVINCE.get(c[ix["Subdomain"]].strip().lower(), None)
            out.append(dict(ref=c[ix["reference"]].strip(), age=age, mtl=mtl, prov=prov))
    return out


# ---------------------------------------------------------------------------
# PLOTTING
# ---------------------------------------------------------------------------
def draw_sample(ax, s, y, yerr=None):
    robust = s["n_len"] >= ROBUST_LENGTHS
    c = DOMAIN_COLORS[s["domain"]]
    ax.errorbar(s["age"], y, xerr=s["age_se"], yerr=yerr, fmt="none", ecolor=c,
                elinewidth=0.8, capsize=0, alpha=0.6, zorder=3)
    ax.scatter(s["age"], y, s=62, marker=SIDE_MARKERS[s["side"]], c=c,
               edgecolors="#1b1b1b", linewidths=0.7 if robust else 0.6,
               linestyles="solid" if robust else (0, (1.5, 1.2)), zorder=5)


def style_axes(ax, ylabel):
    ax.set_xlabel("AFT central age (Ma)", fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.grid(True, color="#dddddd", lw=0.5, zorder=0)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=8.5)


def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    samples, skipped = [], []
    for f in sample_names():
        name = os.path.splitext(os.path.basename(f))[0]
        num = int(re.search(r"(\d+)", name).group(1))
        Ns, Ni, rhod, Nd, L = read_sheet(f)
        if name in EXCLUDE:
            skipped.append(f"{name} (excluded by hand)")
            continue
        if len(L) < MIN_LENGTHS:
            skipped.append(f"{name} (n = {len(L)} lengths)")
            continue
        age, se = central_age(Ns, Ni, rhod, Nd)
        mtl_p, src = projected_mtl(num, age)
        if np.isnan(mtl_p):
            print(f"[warning] {name}: no QTQt _pred.txt with matching age; projected MTL missing")
        samples.append(dict(sample=name, num=num, domain=classify_domain(num),
                            side="west" if num in WEST_OF_TUCANO else "east", age=age,
                            age_se=se, n_len=len(L), mtl=L.mean(), sd=L.std(ddof=1),
                            mtl_p=mtl_p, src=src))
    lit = read_literature(LIT_TSV)

    with open(os.path.join(OUT_DIR, "boomerang_data.csv"), "w", encoding="utf-8") as fh:
        fh.write("sample,domain,side_of_Tucano,central_age_Ma,1sigma_Ma,n_lengths,MTL_measured_um,SD_measured_um,"
                 "MTL_projected_um,projected_source\n")
        for s in samples:
            fh.write(f"{s['sample']},{s['domain']},{s['side']},{s['age']:.1f},{s['age_se']:.1f},{s['n_len']},"
                     f"{s['mtl']:.2f},{s['sd']:.2f},{s['mtl_p']:.2f},\"{s['src']}\"\n")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 5.6))
    xmax = max([s["age"] + s["age_se"] for s in samples] + [l["age"] for l in lit] + [100]) * 1.05

    # (a) this study, projected MTL
    labels = []
    for s in samples:
        if np.isfinite(s["mtl_p"]):
            draw_sample(ax1, s, s["mtl_p"])
            if LABEL_SAMPLES:
                labels.append(ax1.text(s["age"] + 3, s["mtl_p"] + 0.03, str(s["num"]).zfill(3),
                                       fontsize=6.5, color="#444444", zorder=7))
    style_axes(ax1, "c-axis projected MTL (µm)")
    ax1.set_title("(a) This study — c-axis projected lengths", loc="left", fontsize=10.5,
                  fontweight="bold")

    # (b) measured MTL, this study + literature
    refs = sorted({l["ref"] for l in lit})
    for k, ref in enumerate(refs):
        pts = [l for l in lit if l["ref"] == ref]
        mk = LIT_MARKERS[k % len(LIT_MARKERS)]
        cols = [DOMAIN_COLORS.get(p["prov"], LIT_COLOR) for p in pts]
        if mk in LINE_MARKERS:
            ax2.scatter([p["age"] for p in pts], [p["mtl"] for p in pts], s=48, marker=mk,
                        c=cols, linewidths=1.0, alpha=0.75, zorder=2)
        else:
            ax2.scatter([p["age"] for p in pts], [p["mtl"] for p in pts], s=34, marker=mk,
                        facecolors="white", edgecolors=cols, linewidths=0.9, alpha=0.75, zorder=2)
    for s in samples:
        draw_sample(ax2, s, s["mtl"], yerr=None)
    style_axes(ax2, "Measured MTL (µm)")
    ax2.set_title("(b) Measured lengths — this study and regional literature", loc="left",
                  fontsize=10.5, fontweight="bold")

    pts_a = [s["mtl_p"] for s in samples if np.isfinite(s["mtl_p"])]
    pts_b = [s["mtl"] for s in samples] + [l["mtl"] for l in lit]
    for ax, v in ((ax1, pts_a), (ax2, pts_b)):
        ax.set_xlim(0, xmax)
        ax.set_ylim(np.floor((min(v) - 0.3) * 2) / 2, np.ceil((max(v) + 0.3) * 2) / 2)
    if labels:   # move overlapping sample labels apart (optional package adjustText)
        try:
            from adjustText import adjust_text
            adjust_text(labels, ax=ax1, expand=(1.3, 1.6),
                        arrowprops=dict(arrowstyle="-", color="#999999", lw=0.5))
        except ImportError:
            print("[info] 'pip install adjustText' to separate overlapping labels")

    # legend
    h = []
    for d in DOMAIN_ORDER:
        for side in ("west", "east"):
            n = sum(s["domain"] == d and s["side"] == side for s in samples)
            if n:
                h.append(Line2D([0], [0], marker=SIDE_MARKERS[side], ls="", mfc=DOMAIN_COLORS[d],
                                mec="#1b1b1b", ms=8, label=f"{d}, {SIDE_LABELS[side]} (n = {n})"))
    h.append(plt.scatter([], [], s=62, marker="o", c="#bbbbbb", edgecolors="#1b1b1b",
                         linewidths=0.6, linestyles=(0, (1.5, 1.2)),
                         label=f"{MIN_LENGTHS}–{ROBUST_LENGTHS - 1} confined tracks (dashed border)"))
    for k, ref in enumerate(refs):
        mk = LIT_MARKERS[k % len(LIT_MARKERS)]
        h.append(Line2D([0], [0], marker=mk, ls="", mfc="white" if mk not in LINE_MARKERS else LIT_COLOR,
                        mec=LIT_COLOR, mew=1.0, ms=7 if mk not in LINE_MARKERS else 8, label=ref))
    fig.legend(handles=h, loc="lower center", ncol=3, frameon=False, fontsize=8.3,
               bbox_to_anchor=(0.5, 0.03))
    fig.text(0.5, 0.005, f"Horizontal bars: central age ±1σ. Only samples with ≥ {MIN_LENGTHS} confined "
             "tracks. Literature (open symbols, coloured by province) restricted to the study region; literature MTL are measured values.",
             fontsize=7, color="#555555", ha="center")
    n_leg_rows = int(np.ceil(len(h) / 3))
    fig.subplots_adjust(left=0.06, right=0.99, top=0.93, bottom=0.16 + 0.04 * n_leg_rows,
                        wspace=0.18)
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(OUT_DIR, f"boomerang_plot.{ext}"), dpi=300)
    plt.close(fig)

    print(f"{len(samples)} samples plotted; skipped: {', '.join(skipped)}")
    print(f"{len(lit)} literature samples inside {LIT_BBOX}")
    for s in samples:
        print(f"  {s['sample']}  {s['age']:6.1f} Ma  n = {s['n_len']:3d}  MTL {s['mtl']:.2f}  "
              f"MTLp {s['mtl_p']:.2f}  <- {os.path.basename(os.path.dirname(s['src']))}/"
              f"{os.path.basename(s['src'])}")


if __name__ == "__main__":
    main()


## 8. This study versus published data in the study area (Fig. 7)

Central age versus measured MTL for the new samples and the published samples west and east of the RTJ rift. West of the rift, the published data are split at 120 Ma (end of RTJ rifting): kernel density envelope (68%, 95%) for the older group and rounded convex hull for the younger group.

Script: `fig07_boomerang_study_area.py`

In [ ]:
%%writefile fig07_boomerang_study_area.py
"""
4b_boomerang_study_area.py -- comparison figure: this study vs literature inside the study area
==============================================================================================

Same boomerang plot (AFT central age vs MEASURED MTL), but the literature is
restricted to the samples that fall inside the study-area polygon (shapefile
'nova_area', the frame of the geological map) and is summarised by an envelope
(convex hull) drawn around the literature points on each side of the RTJ rift:

  - dashed envelope : literature WEST of the rift
  - solid envelope  : literature EAST of the rift
  - grey crosses    : literature INSIDE the rift (rift fill; not enveloped)

This study's samples are plotted on top (colour = tectonic province, shape =
side of the rift), so one can see whether the new data fall inside, or away
from, the field already covered by the published data.

Reuses the readers and settings of 4_boomerang_plots.py (same folder).
Needs: pip install pyshp shapely
"""

import importlib.util
import os

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.patches import Polygon as MplPolygon
import shapefile
from shapely.geometry import MultiPoint, Point, shape
from scipy.stats import gaussian_kde

HERE = os.path.dirname(os.path.abspath(__file__))
spec = importlib.util.spec_from_file_location("bp", os.path.join(HERE, "fig06_half_boomerang.py"))
bp = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bp)

from config import AREA_SHP as STUDY_AREA_SHP
OUT_DIR = bp.OUT_DIR
SIDE_STYLE = {"W": dict(ls=(0, (5, 3)), label="Literature W of RTJ (envelope)"),
              "E": dict(ls="-", label="Literature E of RTJ (envelope)")}
ENV_FILL = "#d9d9d9"
# envelope type: "kde" = density contours (curved, robust to outliers);
#                "hull" = smoothed convex hull around all points
ENVELOPE = "kde"
KDE_LEVELS = (0.68, 0.95)   # fraction of literature points enclosed (filled, outline)


def read_literature_in_area(path, area):
    fl = lambda s: float(str(s).replace(",", ".")) if str(s).strip() not in ("", "-") else np.nan
    out = []
    with open(path, encoding="utf-8", errors="ignore") as fh:
        hdr = fh.readline().rstrip("\n").split("\t")
        ix = {h: i for i, h in enumerate(hdr)}
        for line in fh:
            c = line.rstrip("\n").split("\t")
            if len(c) < len(hdr) - 1:
                continue
            x, y = fl(c[ix["coord_X"]]), fl(c[ix["coord_Y"]])
            age, mtl = fl(c[ix["AFT_age"]]), fl(c[ix["MTL"]])
            if np.isnan(age) or np.isnan(mtl) or not area.contains(Point(x, y)):
                continue
            out.append(dict(sample=c[ix["Sample"]], ref=c[ix["reference"]].strip(), age=age, mtl=mtl,
                            pos=c[ix["RTJ_position"]].strip() if "RTJ_position" in ix else "?"))
    return out


SPLIT_AGE = 120


def kde_envelope(ax, pts, ls, xlim, ylim):
    """Curved envelopes = contours of a 2-D Gaussian KDE enclosing KDE_LEVELS of the points.
    Age and MTL are standardised before the KDE so both axes weigh the same."""
    a = np.array(pts, float)
    mu, sd = a.mean(0), a.std(0)
    kde = gaussian_kde(((a - mu) / sd).T)
    dens_pts = kde(((a - mu) / sd).T)
    gx, gy = np.meshgrid(np.linspace(*xlim, 300), np.linspace(*ylim, 300))
    g = np.vstack([((gx.ravel() - mu[0]) / sd[0]), ((gy.ravel() - mu[1]) / sd[1])])
    dens = kde(g).reshape(gx.shape)
    # density threshold that leaves the requested fraction of points above it
    thr = sorted(np.quantile(dens_pts, 1 - np.array(KDE_LEVELS)))
    ax.contourf(gx, gy, dens, levels=[thr[1], dens.max() * 1.01], colors=[ENV_FILL], alpha=0.45, zorder=1)
    ax.contour(gx, gy, dens, levels=thr, colors="#555555", linewidths=[0.8, 1.3],
               linestyles=[ls, ls], zorder=1.5)


def smooth_hull(ax, pts, ls):
    """Convex hull with rounded corners (buffer in standardised space)."""
    a = np.array(pts, float)
    mu, sd = a.mean(0), a.std(0)
    hull = MultiPoint([tuple(p) for p in (a - mu) / sd]).convex_hull.buffer(0.25, resolution=32)
    xy = np.array(hull.exterior.coords) * sd + mu
    ax.add_patch(MplPolygon(xy, closed=True, fc=ENV_FILL, alpha=0.35, ec="#555555", lw=1.2,
                            ls=ls, zorder=1))


def main():
    area = shape(shapefile.Reader(STUDY_AREA_SHP).shape(0).__geo_interface__)
    lit = read_literature_in_area(bp.LIT_TSV, area)

    # this study (same selection as 4_boomerang_plots.py)
    import glob
    import re
    samples = []
    for f in bp.sample_names():
        name = os.path.splitext(os.path.basename(f))[0]
        num = int(re.search(r"(\d+)", name).group(1))
        Ns, Ni, rhod, Nd, L = bp.read_sheet(f)
        if name in bp.EXCLUDE or len(L) < bp.MIN_LENGTHS:
            continue
        age, se = bp.central_age(Ns, Ni, rhod, Nd)
        samples.append(dict(sample=name, num=num, domain=bp.classify_domain(num),
                            side="west" if num in bp.WEST_OF_TUCANO else "east",
                            age=age, age_se=se, n_len=len(L), mtl=L.mean()))

    allv = [l["mtl"] for l in lit] + [s["mtl"] for s in samples]
    ylim = (np.floor((min(allv) - 0.3) * 2) / 2, np.ceil((max(allv) + 0.3) * 2) / 2)
    xlim = (0, max([l["age"] for l in lit] + [s["age"] + s["age_se"] for s in samples]) * 1.05)
    inside = [l for l in lit if l["pos"] not in ("W", "E")]
    SIDE_OF = {"W": "west", "E": "east"}
    TITLES = {("W", "E"): "This study vs literature inside the study area",
              ("W",): "West of the RTJ rift — this study vs literature",
              ("E",): "East of the RTJ rift — this study vs literature"}

    def draw_panel(ax, sides):
        both = len(sides) == 2
        split_w = tuple(sides) == ("W",)   # west panel: literature split at SPLIT_AGE (end of RTJ rifting)
        for pos in sides:
            pts = [(l["age"], l["mtl"]) for l in lit if l["pos"] == pos]
            if pos == "W" and split_w:
                old = [p for p in pts if p[0] > SPLIT_AGE]
                young = [p for p in pts if p[0] <= SPLIT_AGE]
                kde_envelope(ax, old, SIDE_STYLE[pos]["ls"], xlim, ylim)
                smooth_hull(ax, young, SIDE_STYLE[pos]["ls"])
                ax.axvline(SPLIT_AGE, color="#2b2a2c", lw=1.0, ls=(0, (4, 3)), zorder=1.6)
                ax.text(SPLIT_AGE + 3, ylim[0] + 0.15, f"{SPLIT_AGE} Ma", fontsize=8, color="#2b2a2c")
                print(f"west literature split at {SPLIT_AGE} Ma: > {SPLIT_AGE} Ma n = {len(old)}, "
                      f"median MTL {np.median([p[1] for p in old]):.2f}; <= {SPLIT_AGE} Ma n = {len(young)}, "
                      f"median MTL {np.median([p[1] for p in young]):.2f}")
            elif len(pts) >= 5 and ENVELOPE == "kde":
                kde_envelope(ax, pts, SIDE_STYLE[pos]["ls"], xlim, ylim)
            elif len(pts) >= 3:
                smooth_hull(ax, pts, SIDE_STYLE[pos]["ls"])
            ax.scatter([p[0] for p in pts], [p[1] for p in pts], s=18,
                       marker="s" if pos == "W" else "o", facecolors="white", edgecolors="#777777",
                       linewidths=0.8, zorder=2)
        if both:
            ax.scatter([l["age"] for l in inside], [l["mtl"] for l in inside], s=26, marker="x",
                       c="#9a9a9a", linewidths=0.9, zorder=2)
        keep = {SIDE_OF[p] for p in sides}
        for s in samples:
            if s["side"] in keep:
                bp.draw_sample(ax, s, s["mtl"])
            else:   # other side: faded, for context only
                ax.scatter(s["age"], s["mtl"], s=40, marker=bp.SIDE_MARKERS[s["side"]],
                           facecolors="none", edgecolors="#bdbdbd", linewidths=0.8, zorder=1.8)
        bp.style_axes(ax, "Measured MTL (µm)")
        ax.set_ylim(*ylim)
        ax.set_xlim(*xlim)
        ax.set_title(TITLES[tuple(sides)], loc="left", fontsize=11, fontweight="bold")

        h = []
        for d in bp.DOMAIN_ORDER:
            for side in ("west", "east"):
                n = sum(s["domain"] == d and s["side"] == side for s in samples)
                if n and side in keep:
                    h.append(Line2D([0], [0], marker=bp.SIDE_MARKERS[side], ls="", mfc=bp.DOMAIN_COLORS[d],
                                    mec="#1b1b1b", ms=8, label=f"{d}, {bp.SIDE_LABELS[side]} (n = {n})"))
        for pos, mk in (("W", "s"), ("E", "o")):
            if pos in sides:
                n = sum(l["pos"] == pos for l in lit)
                h.append(Line2D([0], [0], color="#555555", lw=1.2, ls=SIDE_STYLE[pos]["ls"], marker=mk,
                                mfc="white", mec="#777777", ms=5, label=f"{SIDE_STYLE[pos]['label']} (n = {n})"))
        if split_w:
            h.append(Line2D([0], [0], color="#2b2a2c", lw=1.0, ls=(0, (4, 3)),
                            label=f"{SPLIT_AGE} Ma: end of RTJ rifting (Tavares et al., 2026)"))
            h.append(Patch(fc=ENV_FILL, alpha=0.35, ec="#555555", ls=SIDE_STYLE["W"]["ls"],
                           label=f"Literature W ≤ {SPLIT_AGE} Ma (rounded convex hull)"))
        if both:
            h.append(Line2D([0], [0], marker="x", ls="", color="#9a9a9a", ms=6,
                            label=f"Literature inside the RTJ rift (n = {len(inside)})"))
        else:
            h.append(Line2D([0], [0], marker="o", ls="", mfc="none", mec="#bdbdbd", ms=6,
                            label="This study, other side of the rift"))
        ax.legend(handles=h, loc="upper right", fontsize=7.8, frameon=False)

    note = ("Literature: samples inside the study-area polygon with published MTL "
            "(measured lengths), grouped with the RTJ rift polygon.\n"
            + ("Envelopes = 2-D kernel density contours enclosing "
               + " and ".join(f"{int(l * 100)}%" for l in KDE_LEVELS)
               + " of the literature points (filled = inner); west of the rift, the KDE is drawn for ages > 120 Ma only." if ENVELOPE == "kde"
               else "Envelopes = convex hulls with rounded corners."))

    # single-panel figures: both sides, west only, east only
    for sides, tag in ((("W", "E"), ""), (("W",), "_W"), (("E",), "_E")):
        fig, ax = plt.subplots(figsize=(8.6, 6.4))
        draw_panel(ax, sides)
        fig.text(0.01, 0.005, note, fontsize=7, color="#555555")
        fig.tight_layout(rect=[0, 0.04, 1, 1])
        for ext in ("pdf", "png"):
            fig.savefig(os.path.join(OUT_DIR, f"boomerang_study_area{tag}.{ext}"), dpi=300)
        plt.close(fig)

    # west and east side by side (same axes)
    fig, axs = plt.subplots(1, 2, figsize=(16, 6.4), sharey=True)
    draw_panel(axs[0], ("W",))
    draw_panel(axs[1], ("E",))
    axs[0].set_title("(a) " + TITLES[("W",)], loc="left", fontsize=11, fontweight="bold")
    axs[1].set_title("(b) " + TITLES[("E",)], loc="left", fontsize=11, fontweight="bold")
    axs[1].set_ylabel("")
    fig.text(0.01, 0.005, note, fontsize=7, color="#555555")
    fig.tight_layout(rect=[0, 0.04, 1, 1])
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(OUT_DIR, f"boomerang_study_area_W_E.{ext}"), dpi=300)
    plt.close(fig)

    # numbers for the text
    def summ(v):
        return f"median {np.median(v):.0f} Ma (IQR {np.percentile(v, 25):.0f}-{np.percentile(v, 75):.0f})" if len(v) else "-"
    for side, pos in (("west", "W"), ("east", "E")):
        own = [s["age"] for s in samples if s["side"] == side]
        ref = [l["age"] for l in lit if l["pos"] == pos]
        own_m = [s["mtl"] for s in samples if s["side"] == side]
        ref_m = [l["mtl"] for l in lit if l["pos"] == pos]
        print(f"{side:4s}  this study n={len(own):2d} {summ(own)}, MTL {np.median(own_m):.2f} | "
              f"literature n={len(ref):2d} {summ(ref)}, MTL {np.median(ref_m) if ref_m else np.nan:.2f}")
    print(f"inside rift: n={len(inside)}")
    print("figures: boomerang_study_area.png, _W.png, _E.png, _W_E.png ->", OUT_DIR)


if __name__ == "__main__":
    main()


## 9. Age versus distance to the coast and to the rift (Fig. 8)

Distance of each sample to the present coastline and signed distance to the RTJ rift axis; OLS fits (distance to the coast, step function at the rift, both) and Spearman correlations within each side. Uses the central ages written by Section 2.

Script: `fig08_age_vs_distance.py`

In [ ]:
%%writefile fig08_age_vs_distance.py
"""fig08_age_vs_distance.py -- AFT central age vs (a) distance to the present coastline and
(b) signed distance to the axis of the Reconcavo-Tucano-Jatoba (RTJ) rift (W negative, E positive).

Geometry (all distances computed in SIRGAS 2000 / UTM 24S, EPSG:31984):
  coastline  landward edge of the coastal water units of the 1:5,000,000 geomorphological map of
             Brazil (IBGE), clipped to NE Brazil (42.5-34 W, 16-5 S)
  rift axis  centre line of the RTJ rift polygon: mid-point between its western and eastern edges,
             sampled every 0.01 deg of latitude south of 9.05 S (Tucano and Reconcavo segments)
  side       new samples: Table 1; literature: column RTJ_position (W / E) of the compilation;
             samples inside the rift are excluded.
Statistics: OLS fits of age on distance to the coast, on a step function (one mean per side of the
rift) and on both; R^2 for the new data (n = 27) and for all samples outside the rift. Spearman's rho
between age and distance to the axis within each side.
Inputs: COUNTS-derived central ages (output of figS1_radial_plots.py), sample coordinates
(data workbook: sheets 'Samples' and 'Literature'), shapefiles.
"""
import csv, os
import numpy as np
import openpyxl
import shapefile
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pyproj import Transformer
from scipy.stats import spearmanr
from shapely.geometry import LineString, Point, box, shape
from shapely.ops import unary_union, transform
from config import SAMPLES_XLSX, SAMPLES_SHEET, LIT_TSV, RTJ_SHP, COAST_SHP, OUT

plt.rcParams["font.family"] = ["Montserrat", "DejaVu Sans"]
plt.rcParams["pdf.fonttype"] = 42
TO_UTM = Transformer.from_crs(4674, 31984, always_xy=True).transform
SFC_SAMPLES = {1, 5, 9, 10, 16, 18, 21, 101, 104, 106, 108, 112, 114, 116, 120}
COL = {"W": "#a6423a", "E": "#2f6fa8"}      # colour = side of the rift
MK = {"SFC": "s", "BP": "o"}               # shape = province
YOUNG_W = 120                               # literature W of the rift younger than this: open diamonds (cf. Fig. 7a)
RIFT_HALF_WIDTH = 30                        # km, shaded band around the axis (schematic)


def rift_axis(rift, lat_max=-9.05, step=0.01):
    pts = []
    for y in np.arange(rift.bounds[1] + step / 2, lat_max, step):
        seg = rift.intersection(LineString([(-45, y), (-30, y)]))
        if seg.is_empty:
            continue
        xs = [c[0] for g in getattr(seg, "geoms", [seg]) for c in g.coords]
        pts.append(((min(xs) + max(xs)) / 2, y))
    return transform(TO_UTM, LineString(pts))


def coastline(path, bbox=(-42.5, -16, -34, -5)):
    """Coastline = landward edge of the coastal water units of the geomorphological map
    ('Massa Dagua Costeira': territorial sea, contiguous zone, EEZ); inland water bodies
    ('Massa Dagua Continental') are ignored."""
    bb = box(*bbox)
    sea = unary_union([shape(sr.shape.__geo_interface__).buffer(0)
                       for sr in shapefile.Reader(path, encoding="latin1").iterShapeRecords()
                       if str(sr.record["NOM_UNIDAD"]).startswith("Massa Dagua Costeira")]).intersection(bb)
    return transform(TO_UTM, sea.boundary.difference(bb.exterior.buffer(1e-4)))


def ols_r2(cols, y):
    X = np.column_stack([np.ones(len(y))] + cols)
    b = np.linalg.lstsq(X, y, rcond=None)[0]
    return 1 - ((y - X @ b) ** 2).sum() / ((y - y.mean()) ** 2).sum(), b


def main():
    rift = shape(shapefile.Reader(RTJ_SHP).shape(0).__geo_interface__)
    axis, coast = rift_axis(rift), coastline(COAST_SHP)

    ages = {r["sample"]: (float(r["central_age_Ma"]), float(r["1sigma_Ma"]))
            for r in csv.DictReader(open(os.path.join(OUT, "AFT_central_age_summary.csv"), encoding="utf-8"))}
    ws = openpyxl.load_workbook(SAMPLES_XLSX, read_only=True, data_only=True)[SAMPLES_SHEET]
    it = ws.iter_rows(values_only=True); hdr = list(next(it))
    new = []
    for r in it:
        d = dict(zip(hdr, r))
        if not d.get("Sample"):
            continue
        p = Point(*TO_UTM(float(d["Lon"]), float(d["Lat"])))
        sgn = -1 if d["Side"] == "W" else 1
        num = int(str(d["Sample"])[4:])
        new.append(dict(n=d["Sample"], age=ages[d["Sample"]][0], err=ages[d["Sample"]][1], side=d["Side"],
                        prov="SFC" if num in SFC_SAMPLES else "BP",
                        dc=coast.distance(p) / 1e3, da=sgn * axis.distance(p) / 1e3))
    lit = []
    fl = lambda v: float(str(v).replace(",", "."))
    for l in csv.DictReader(open(LIT_TSV, encoding="utf-8", errors="ignore"), delimiter="\t"):
        if l.get("RTJ_position") not in ("W", "E"):
            continue
        try:
            x, y, a = fl(l["coord_X"]), fl(l["coord_Y"]), fl(l["AFT_age"])
        except ValueError:
            continue
        p = Point(*TO_UTM(x, y))
        sgn = -1 if l["RTJ_position"] == "W" else 1
        lit.append(dict(age=a, side=l["RTJ_position"], dc=coast.distance(p) / 1e3, da=sgn * axis.distance(p) / 1e3))

    # statistics
    stats = {}
    for tag, S in (("new", new), ("all", new + lit)):
        a = np.array([s["age"] for s in S]); dc = np.array([s["dc"] for s in S])
        e = np.array([s["da"] > 0 for s in S], float)
        stats[tag] = dict(n=len(S), coast=ols_r2([dc], a), step=ols_r2([e], a), both=ols_r2([e, dc], a))
        print(f"{tag}: n = {len(S)}; R2 coast = {stats[tag]['coast'][0]:.2f}; R2 step = {stats[tag]['step'][0]:.2f}; "
              f"R2 step + coast = {stats[tag]['both'][0]:.2f} (coast coefficient {stats[tag]['both'][1][2]:.3f} Ma/km)")
    for tag, S in (("new", new), ("literature", lit)):
        for side, sel in (("W", lambda s: s["da"] < 0), ("E", lambda s: s["da"] > 0)):
            T = [s for s in S if sel(s)]
            rho = spearmanr([abs(s["da"]) for s in T], [s["age"] for s in T])[0]
            print(f"{tag} {side}: n = {len(T)}, Spearman rho(age, distance to axis) = {rho:.2f}")
    with open(os.path.join(OUT, "age_vs_distance_data.csv"), "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh); w.writerow(["sample", "age_Ma", "side", "dist_coast_km", "signed_dist_axis_km"])
        for s in new:
            w.writerow([s["n"], s["age"], s["side"], round(s["dc"], 1), round(s["da"], 1)])

    # figure
    fig, axs = plt.subplots(1, 2, figsize=(16, 6.4), sharey=True)
    young = [l for l in lit if l["side"] == "W" and l["age"] <= YOUNG_W]
    other = [l for l in lit if l not in young]
    for ax, key, xl in ((axs[0], "dc", "Distance to present coastline (km)"),
                        (axs[1], "da", "Distance to RTJ rift axis (km)  ·  W negative, E positive")):
        ax.scatter([l[key] for l in other], [l["age"] for l in other], s=34, facecolors="white",
                   edgecolors="#9a9a9a", linewidths=0.9, zorder=2)
        ax.scatter([l[key] for l in young], [l["age"] for l in young], s=70, marker="D", facecolors="white",
                   edgecolors=COL["E"], linewidths=1.6, zorder=2.5)
        for s_ in new:
            ax.errorbar(s_[key], s_["age"], yerr=s_["err"], fmt="none", ecolor=COL[s_["side"]], elinewidth=1,
                        alpha=0.6, zorder=3)
            ax.scatter(s_[key], s_["age"], s=150, marker=MK[s_["prov"]], c=COL[s_["side"]], edgecolors="#1b1b1b",
                       linewidths=1.0, zorder=4)
        ax.set_xlabel(xl, fontsize=13)
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(True, color="#e6e6e6", lw=0.6, zorder=0)
        ax.tick_params(labelsize=11)
    axs[0].set_ylabel("AFT central age (Ma)", fontsize=13)
    axs[0].set_ylim(0, 380)
    allp = new + lit
    a = np.array([p["age"] for p in allp]); dc = np.array([p["dc"] for p in allp]); da = np.array([p["da"] for p in allp])
    b = stats["all"]["coast"][1]; xx = np.linspace(dc.min(), dc.max(), 50)
    axs[0].plot(xx, b[0] + b[1] * xx, color="#2b2a2c", lw=1.5, ls="--", zorder=5)
    for sgn, sel in ((-1, da < 0), (1, da > 0)):
        lo, hi = (da[sel].min(), -20) if sgn < 0 else (20, da[sel].max())
        axs[1].plot([lo, hi], [a[sel].mean()] * 2, color="#2b2a2c", lw=1.5, ls="--", zorder=5)
    axs[1].axvspan(-RIFT_HALF_WIDTH, RIFT_HALF_WIDTH, color="#F1DCA7", alpha=0.5, lw=0, zorder=0)
    axs[1].axvline(0, color="#2b2a2c", lw=1.2, ls=(0, (4, 3)), zorder=1)
    axs[1].text(0, 370, "RTJ\nrift", ha="center", va="top", fontsize=11, color="#8a6d2b", fontweight=600)
    axs[0].text(0.97, 0.04, f"linear fit, all data: R² = {stats['all']['coast'][0]:.2f}", transform=axs[0].transAxes,
                ha="right", fontsize=11.5)
    axs[1].text(0.97, 0.04, f"step at the rift (mean per side), all data: R² = {stats['all']['step'][0]:.2f}",
                transform=axs[1].transAxes, ha="right", fontsize=11.5)
    for ax, t in zip(axs, ("(a)", "(b)")):
        ax.set_title(t, loc="left", fontsize=14, fontweight=700)
    h = [Line2D([0], [0], marker="o", ls="", mfc=COL["W"], mec="#1b1b1b", ms=10, label="This study, west of the RTJ"),
         Line2D([0], [0], marker="o", ls="", mfc=COL["E"], mec="#1b1b1b", ms=10, label="This study, east of the RTJ"),
         Line2D([0], [0], marker="D", ls="", mfc="white", mec=COL["E"], mew=1.6, ms=8, label=f"Literature W, ≤ {YOUNG_W} Ma"),
         Line2D([0], [0], marker="s", ls="", mfc="white", mec="#1b1b1b", ms=9, label="SFC"),
         Line2D([0], [0], marker="o", ls="", mfc="white", mec="#1b1b1b", ms=9, label="BP"),
         Line2D([0], [0], marker="o", ls="", mfc="white", mec="#9a9a9a", ms=7, label="Literature (outside the rift)")]
    fig.legend(handles=h, loc="lower center", ncol=6, frameon=False, fontsize=11, bbox_to_anchor=(0.5, 0.0))
    fig.subplots_adjust(left=0.06, right=0.99, top=0.93, bottom=0.2, wspace=0.06)
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(OUT, f"age_vs_distance.{ext}"), dpi=300)
    plt.close(fig)


if __name__ == "__main__":
    main()


## 10. Expected t–T paths west and east of the RTJ rift (Fig. 9)

Expected paths of the 23 modelled samples, mean path of each side of the rift and APAZ (60–120 °C).

Script: `fig09_tT_paths_by_rift_side.py`

In [ ]:
%%writefile fig09_tT_paths_by_rift_side.py
"""
6_tT_paths_by_rift_side.py  --  QTQt expected t-T paths, west vs east of the Tucano Basin
=======================================================================================

Three panels (same colours as the AFT-age kriging map: west = red, east = blue):
  (a) west of the RTJ rift: individual expected paths (thin) + group mean (bold)
      + min-max envelope of the individual paths
  (b) east of the RTJ rift: same
  (c) both group means and envelopes together

Samples whose tectonic province differs from the rest of their side are drawn
dashed and labelled (TFS-082 = Borborema Province west of the rift; TFS-001,
-005, -009 = São Francisco Craton east of the rift).

Input: the folder with "TFS-XXX expected Tt file.txt" (QTQt export: line 1
header, line 2 = number of points, then  t (Ma)  T  T_lo  T_hi).
TFS-021 is skipped (no valid track lengths).
"""

import glob
import os
import re

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.formatter.use_locale"] = False

from config import TT_DIR, OUT
OUT_DIR = OUT

WEST_OF_TUCANO = {82, 101, 104, 106, 108, 112, 114, 116, 120}
SFC_SAMPLES = {1, 5, 9, 10, 16, 18, 21, 101, 104, 106, 108, 112, 114, 116, 120}
EXCLUDE = {21}
SIDE_COLORS = {"west": "#b2302b", "east": "#2f6fa8"}      # red / blue, as in the kriging maps
SIDE_NAMES = {"west": "West of the Tucano Basin", "east": "East of the Tucano Basin"}
APAZ = (60, 120)
XMAX = 480
TMAX_PLOT = 135
MIN_FRACTION = 0.6        # group mean drawn only where >= 60% of the group's paths exist
SMOOTH_MYR = 21           # running-mean window for the group mean (removes steps where paths start)


def read_tt(path):
    a = []
    with open(path, errors="ignore") as fh:
        for line in fh.readlines()[2:]:
            p = line.split()
            try:
                v = [float(x) for x in p]
            except ValueError:
                continue
            if len(v) >= 2:
                a.append(v[:4] + [np.nan] * (4 - len(v)))
    a = np.array(a)
    o = np.argsort(a[:, 0])
    return a[o, 0], a[o, 1]


def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    S = []
    for f in sorted(glob.glob(os.path.join(TT_DIR, "TFS-*expected Tt file.txt"))):
        num = int(re.search(r"TFS-(\d+)", os.path.basename(f)).group(1))
        if num in EXCLUDE:
            continue
        t, T = read_tt(f)
        side = "west" if num in WEST_OF_TUCANO else "east"
        odd = (side == "west") != (num in SFC_SAMPLES)   # province differs from its side
        S.append(dict(num=num, side=side, t=t, T=T, odd=odd))

    grid = np.arange(0, XMAX + 1, 1.0)

    def stack(side):
        rows = []
        for s in S:
            if s["side"] == side:
                g = np.interp(grid, s["t"], s["T"], left=np.nan, right=np.nan)
                g[grid > s["t"].max()] = np.nan
                rows.append(g)
        return np.vstack(rows)

    stats = {}
    for side in ("west", "east"):
        A = stack(side)
        n = np.sum(np.isfinite(A), axis=0)
        with np.errstate(all="ignore"):
            mean, lo, hi = np.nanmean(A, 0), np.nanmin(A, 0), np.nanmax(A, 0)
        ok = n >= max(2, MIN_FRACTION * A.shape[0])
        mean[~ok] = np.nan
        if SMOOTH_MYR > 1:
            k = np.ones(SMOOTH_MYR) / SMOOTH_MYR
            v = np.where(ok, mean, 0.0)
            w = np.convolve(ok.astype(float), k, "same")
            sm = np.convolve(v, k, "same") / np.where(w > 0, w, np.nan)
            mean = np.where(ok, sm, np.nan)
        stats[side] = dict(mean=mean, lo=lo, hi=hi, n=A.shape[0])

    fig, axs = plt.subplots(1, 3, figsize=(18, 6.2), sharey=True)
    for ax in axs:
        ax.axhspan(*APAZ, color="#e8e4da", zorder=0, lw=0)
        ax.text(XMAX - 5, (APAZ[0] + APAZ[1]) / 2, "APAZ", style="italic", color="#8a8370",
                fontsize=9, ha="left", va="center")
        ax.set_xlim(XMAX, 0)
        ax.set_ylim(TMAX_PLOT, 0)
        ax.set_xlabel("Time (Ma)", fontsize=10)
        ax.grid(True, color="#dddddd", lw=0.5, zorder=0)
        ax.spines[["top", "right"]].set_visible(False)
        ax.tick_params(labelsize=8.5)
    axs[0].set_ylabel("Temperature (°C)", fontsize=10)

    for ax, side, letter in ((axs[0], "west", "a"), (axs[1], "east", "b")):
        c = SIDE_COLORS[side]
        st = stats[side]
        ax.fill_between(grid, st["lo"], st["hi"], color=c, alpha=0.13, lw=0, zorder=1)
        for s in S:
            if s["side"] != side:
                continue
            ax.plot(s["t"], s["T"], color=c, lw=1.6 if s["odd"] else 0.9,
                    ls=(0, (4, 2)) if s["odd"] else "-", alpha=0.95 if s["odd"] else 0.45, zorder=2)
            if s["odd"]:
                i = int(np.argmax(s["T"]))
                ax.annotate(f"TFS-{s['num']:03d}", (s["t"][i], s["T"][i]), xytext=(4, -6),
                            textcoords="offset points", fontsize=7.5, color=c, fontweight="bold")
        ax.plot(grid, st["mean"], color=c, lw=3, zorder=4)
        ax.set_title(f"({letter}) {SIDE_NAMES[side]} (n = {st['n']})", loc="left",
                     fontsize=10.5, fontweight="bold")

    ax = axs[2]
    for side in ("west", "east"):
        c, st = SIDE_COLORS[side], stats[side]
        ax.fill_between(grid, st["lo"], st["hi"], color=c, alpha=0.13, lw=0, zorder=1)
        ax.plot(grid, st["mean"], color=c, lw=3, zorder=4)
    ax.set_title("(c) Group means compared", loc="left", fontsize=10.5, fontweight="bold")

    h = [Line2D([0], [0], color=SIDE_COLORS["west"], lw=3, label=f"{SIDE_NAMES['west']}: mean path"),
         Line2D([0], [0], color=SIDE_COLORS["east"], lw=3, label=f"{SIDE_NAMES['east']}: mean path"),
         Line2D([0], [0], color="#777777", lw=0.9, alpha=0.6, label="Individual expected path (QTQt)"),
         Line2D([0], [0], color="#777777", lw=0, label=f"Mean: {SMOOTH_MYR}-Myr running mean, where ≥ {int(MIN_FRACTION*100)}% of paths exist"),
         Line2D([0], [0], color="#777777", lw=1.6, ls=(0, (4, 2)),
                label="Province differs from its side (TFS-082 Borborema W; TFS-001/005/009/010/016 SFC E)"),
         Patch(color="#999999", alpha=0.25, label="Envelope of individual paths (min–max)"),
         Patch(color="#e8e4da", label=f"APAZ ({APAZ[0]}–{APAZ[1]} °C)")]
    fig.legend(handles=h, loc="lower center", ncol=3, frameon=False, fontsize=8.5, bbox_to_anchor=(0.5, 0.0))
    fig.subplots_adjust(left=0.05, right=0.99, top=0.93, bottom=0.2, wspace=0.07)
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(OUT_DIR, f"tT_paths_W_vs_E.{ext}"), dpi=300)
    plt.close(fig)

    # summary: peak temperature and age of the mean path, and when it leaves the APAZ
    for side in ("west", "east"):
        m = stats[side]["mean"]
        ok = np.isfinite(m)
        g, mm = grid[ok], m[ok]
        ip = int(np.argmax(mm))
        below = np.where((g < g[ip]) & (mm <= APAZ[0]))[0]
        exit_age = g[below].max() if len(below) else np.nan
        print(f"{side}: n = {stats[side]['n']}, mean path drawn from {g.max():.0f} Ma "
              f"({mm[-1] if False else mm[np.argmax(g)]:.0f} °C); mean leaves the APAZ (<{APAZ[0]} °C) at ~{exit_age:.0f} Ma")
    print("figure ->", OUT_DIR)


if __name__ == "__main__":
    main()


## 11. Reheating test (Supplementary Table S2)

Maximum post-rift reheating and temperature at 80 Ma on 5000 posterior paths per sample.

Script: `tabS2_reheating_check.py`

In [ ]:
%%writefile tabS2_reheating_check.py
"""
7_reheating_check.py  --  does the QTQt posterior allow / require post-rift reheating?
=====================================================================================
1) Prints the prior settings of every chain used (header line 9: "No reheating = 0"
   means reheating was ALLOWED; 1 = forbidden).
2) For each sample (every 20th post-burn-in model, 5000 models) computes:
   - reheat: largest temperature increase after 140 Ma along each model path
             (T(t) minus the running minimum since 140 Ma), median and 95th percentile
   - P_reheat_gt10 / gt20: fraction of models reheated by > 10 / > 20 degC after 140 Ma
   - T80: temperature at 80 Ma (Campanian, Japsen et al. 2012 maximum burial), median,
          95th percentile and P(T80 > 60 degC)
Output: reheating_check.csv in OUT_DIR.
Chains: FINAL folder has priority over 'projected round 2'; TFS-021 skipped.
"""
import glob, os, re, numpy as np, csv
from config import QTQT_DIRS, OUT as OUT_DIR
W={82,101,104,106,108,112,114,116,120}
files=[f for d in QTQT_DIRS[::-1] for f in glob.glob(os.path.join(d,"*","*.txt"))
       if not re.search(r"_pred|expected|Documentation",f)]
chosen={}
for f in files:
    num=int(re.search(r"[\\/]TFS-(\d+)[\\/]",f).group(1))
    if num==21: continue
    chosen[num]=f
T0=140; grid=np.arange(0,501,1.0); out=[]
for num in sorted(chosen):
    f=chosen[num]; ms=[]
    with open(f,errors="ignore") as fh: h=[next(fh).strip() for _ in range(9)]
    print(f"TFS-{num:03d}: prior {h[6]} | {h[8]} | {os.path.basename(f)}")
    with open(f,errors="ignore") as fh:
        c=False
        for i,l in enumerate(fh):
            if l.startswith("CHAIN END"): break
            if l.startswith("CHAIN"): c=True; continue
            if c and i%20==0:
                v=[float(x) for x in l.split()]; n=int(v[3]); p=np.array(v[4:4+2*(n+1)]).reshape(-1,2)
                p=p[np.argsort(p[:,0])]; ms.append(p)
    rh=[];t80=[];t120=[]
    for p in ms:
        Tg=np.interp(grid,p[:,0],p[:,1],left=np.nan,right=np.nan)
        seg=Tg[(grid<=T0)][::-1]          # from 140 Ma toward present
        seg=seg[np.isfinite(seg)]
        rh.append(np.max(seg-np.minimum.accumulate(seg)) if len(seg) else np.nan)
        t80.append(np.interp(80,p[:,0],p[:,1])); t120.append(np.interp(120,p[:,0],p[:,1]))
    rh=np.array(rh);t80=np.array(t80)
    out.append(dict(sample=f"TFS-{num:03d}",side="W" if num in W else "E",n_models=len(ms),
        reheat_median=round(np.nanmedian(rh),1),reheat_p95=round(np.nanpercentile(rh,95),1),
        P_reheat_gt10=round(np.mean(rh>10),2),P_reheat_gt20=round(np.mean(rh>20),2),
        T80_median=round(np.median(t80),0),T80_p95=round(np.percentile(t80,95),0),P_T80_gt60=round(np.mean(t80>60),2)))
k=list(out[0]); w=csv.DictWriter(open(os.path.join(OUT_DIR,"reheating_check.csv"),"w",newline=""),k); w.writeheader(); w.writerows(out)
for o in out: print(*[o[x] for x in k],sep="\t")


## Run all

Runs the scripts in the order required by their dependencies (the half-boomerang module is imported by the age–elevation and Fig. 7 scripts; the posterior rates use the breakpoint output). Outputs are written to the `output` folder.

In [ ]:
import glob, os, re
from config import QTQT_DIRS

# QTQt chain files are not in the public record (available on request).
# Sections 6 and 11 need them and are skipped when they are missing.
chains = [f for d in QTQT_DIRS for f in glob.glob(os.path.join(d, "*", "*.txt"))
          if not re.search(r"_pred|expected|Documentation", f)]

%run prep_classify_RTJ_position.py
%run figS1_radial_plots.py
%run fig06_half_boomerang.py
%run figS2_age_elevation.py
%run figS3_track_length_histograms.py
%run figS5a_breakpoints.py
if chains:
    %run figS5b_posterior_rates.py
else:
    print("QTQt chain files not found: section 6 (posterior rates) skipped")
%run fig07_boomerang_study_area.py
%run fig08_age_vs_distance.py
%run fig09_tT_paths_by_rift_side.py
if chains:
    %run tabS2_reheating_check.py
else:
    print("QTQt chain files not found: section 11 (reheating test) skipped")
